In [1]:

import pandas as pd
import numpy as np
import os
import json
import zipfile
from pathlib import Path
from datetime import datetime, timezone

print("Real Madrid Analytics")
print("Phase 4.1 — Data Ingestion")
print("Libraries successfully imported.")


Real Madrid Analytics
Phase 4.1 — Data Ingestion
Libraries successfully imported.


In [3]:

from google.colab import files

uploaded = files.upload()

zip_files = [
    name for name in uploaded.keys()
    if name.lower().endswith(".zip")
]

if len(zip_files) != 1:
    raise ValueError(
        "Please upload exactly one updated master dataset ZIP."
    )

ZIP_PATH = zip_files[0]

PROJECT_DIR = Path("/content/real_madrid_project")
CLEANED_DIR = PROJECT_DIR / "Data" / "Cleaned"
OUTPUT_DIR = PROJECT_DIR / "Data" / "Processed" / "01_ingestion"

CLEANED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    archive.extractall(CLEANED_DIR)

print("Master datasets extracted successfully.")
print("Location:", CLEANED_DIR)


Saving Real_Madrid_Master_Datasets_2026-11-18_UPDATED.zip to Real_Madrid_Master_Datasets_2026-11-18_UPDATED.zip
Master datasets extracted successfully.
Location: /content/real_madrid_project/Data/Cleaned


In [4]:

DATASETS = {
    "matches": {
        "file": "Real_Madrid_All_Matches.csv",
        "required": [
            "match_id", "season",
            "match_date", "competition"
        ],
        "key": "match_id"
    },

    "match_team_stats": {
        "file": "Real_Madrid_Match_Team_Stats.csv",
        "required": [
            "match_id", "season",
            "match_date", "competition"
        ],
        "key": "match_id"
    },

    "player_season_stats": {
        "file": "Real_Madrid_Player_Season_Stats.csv",
        "required": ["player_name", "season"],
        "key": None
    },

    "player_competition_stats": {
        "file": "Real_Madrid_Player_Competition_Stats.csv",
        "required": [
            "player_name", "season", "competition"
        ],
        "key": None
    },

    "player_match_stats": {
        "file": "Real_Madrid_Player_Match_Stats.csv",
        "required": [
            "match_id", "player_name", "season"
        ],
        "key": None,
        "allow_empty": True
    },

    "transfers": {
        "file": "Real_Madrid_Transfers.csv",
        "required": [
            "transfer_id", "player_name",
            "season", "direction"
        ],
        "key": "transfer_id"
    },

    "player_id_crosswalk": {
        "file": "Real_Madrid_Player_ID_Crosswalk.csv",
        "required": [
            "player_id", "player_key_candidate"
        ],
        "key": "player_id"
    },

    "financials": {
        "file": "Real_Madrid_Financials.csv",
        "required": [
            "reporting_period", "metric_name",
            "value_original", "data_type", "entity_scope"
        ],
        "key": None
    },

    "trophies": {
        "file": "Real_Madrid_Trophies.csv",
        "required": [
            "trophy_id", "season", "competition"
        ],
        "key": "trophy_id"
    },

    "la_liga_standings": {
        "file": "Real_Madrid_La_Liga_Standings.csv",
        "required": [
            "season", "team", "final_position"
        ],
        "key": None
    }
}

print("Datasets specified:", len(DATASETS))


Datasets specified: 10


In [5]:

def find_dataset(filename):

    matches = list(CLEANED_DIR.rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"Dataset not found: {filename}"
        )

    updated = [
        p for p in matches
        if "UPDATED" in str(p).upper()
    ]

    if len(updated) == 1:
        return updated[0]

    if len(matches) == 1:
        return matches[0]

    raise ValueError(
        f"Multiple copies found for {filename}. "
        "Remove ambiguity by keeping one approved copy."
    )


datasets = {}
manifest = []

for name, spec in DATASETS.items():

    filename = spec["file"]

    try:
        path = find_dataset(filename)

        df = pd.read_csv(
            path,
            encoding="utf-8-sig",
            low_memory=False
        )

        missing_columns = [
            col for col in spec["required"]
            if col not in df.columns
        ]

        duplicate_keys = None

        if spec["key"] and spec["key"] in df.columns:
            duplicate_keys = int(
                df[spec["key"]].duplicated(keep=False).sum()
            )

        datasets[name] = df

        if missing_columns:
            status = "SCHEMA ERROR"

        elif len(df) == 0 and spec.get("allow_empty"):
            status = "EMPTY TEMPLATE"

        elif len(df) == 0:
            status = "UNEXPECTED EMPTY"

        else:
            status = "LOADED"

        manifest.append({
            "dataset": name,
            "filename": filename,
            "rows": len(df),
            "columns": len(df.columns),
            "missing_cells": int(df.isna().sum().sum()),
            "duplicate_key_rows": duplicate_keys,
            "missing_required_columns": ", ".join(missing_columns),
            "status": status
        })

        print(f"{name}: {len(df)} rows — {status}")

    except Exception as e:

        manifest.append({
            "dataset": name,
            "filename": filename,
            "rows": None,
            "columns": None,
            "missing_cells": None,
            "duplicate_key_rows": None,
            "missing_required_columns": str(e),
            "status": "FAILED"
        })

        print(f"ERROR loading {name}: {e}")

print("\nIngestion operation completed.")


matches: 630 rows — LOADED
match_team_stats: 630 rows — LOADED
player_season_stats: 386 rows — LOADED
player_competition_stats: 345 rows — LOADED
player_match_stats: 0 rows — EMPTY TEMPLATE
transfers: 214 rows — LOADED
player_id_crosswalk: 158 rows — LOADED
financials: 204 rows — LOADED
trophies: 22 rows — LOADED
la_liga_standings: 11 rows — LOADED

Ingestion operation completed.


In [6]:

manifest_df = pd.DataFrame(manifest)

print("REAL MADRID DATA INGESTION REPORT")
print("=" * 45)

display(manifest_df)

print("\nTotal datasets:", len(manifest_df))

print(
    "Successfully loaded:",
    manifest_df["status"].isin(
        ["LOADED", "EMPTY TEMPLATE"]
    ).sum()
)

print(
    "Failed:",
    (manifest_df["status"] == "FAILED").sum()
)

print(
    "Schema errors:",
    (manifest_df["status"] == "SCHEMA ERROR").sum()
)


REAL MADRID DATA INGESTION REPORT


,dataset,filename,rows,columns,missing_cells,duplicate_key_rows,missing_required_columns,status
0,matches,Real_Madrid_All_Matches.csv,630,47,7834,0.0,,LOADED
1,match_team_stats,Real_Madrid_Match_Team_Stats.csv,630,30,2844,0.0,,LOADED
2,player_season_stats,Real_Madrid_Player_Season_Stats.csv,386,29,945,NaN,,LOADED
3,player_competition_stats,Real_Madrid_Player_Competition_Stats.csv,345,115,20136,NaN,,LOADED
4,player_match_stats,Real_Madrid_Player_Match_Stats.csv,0,31,0,NaN,,EMPTY TEMPLATE
5,transfers,Real_Madrid_Transfers.csv,214,25,1272,0.0,,LOADED
6,player_id_crosswalk,Real_Madrid_Player_ID_Crosswalk.csv,158,6,0,0.0,,LOADED
7,financials,Real_Madrid_Financials.csv,204,14,205,NaN,,LOADED
8,trophies,Real_Madrid_Trophies.csv,22,12,4,0.0,,LOADED
9,la_liga_standings,Real_Madrid_La_Liga_Standings.csv,11,16,0,NaN,,LOADED



Total datasets: 10
Successfully loaded: 10
Failed: 0
Schema errors: 0


In [7]:

manifest_path = OUTPUT_DIR / "ingestion_manifest.csv"

manifest_df.to_csv(
    manifest_path,
    index=False,
    encoding="utf-8-sig"
)

summary = {
    "project": "Real Madrid: The Business of Football",
    "phase": "4.1 — Data Ingestion",
    "run_date_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "datasets_expected": len(DATASETS),

    "datasets_loaded": int(
        manifest_df["status"].isin(
            ["LOADED", "EMPTY TEMPLATE"]
        ).sum()
    ),

    "datasets_failed": int(
        (manifest_df["status"] == "FAILED").sum()
    ),

    "datasets_with_schema_errors": int(
        (manifest_df["status"] == "SCHEMA ERROR").sum()
    ),

    "source_files_modified": False
}

summary_path = OUTPUT_DIR / "ingestion_summary.json"

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=4)

print("Reports successfully saved.")
print(manifest_path)
print(summary_path)


Reports successfully saved.
/content/real_madrid_project/Data/Processed/01_ingestion/ingestion_manifest.csv
/content/real_madrid_project/Data/Processed/01_ingestion/ingestion_summary.json


In [8]:

from google.colab import files

files.download(str(manifest_path))
files.download(str(summary_path))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:

from pathlib import Path
import pandas as pd
import numpy as np

AUDIT_DIR = Path(
    "/content/real_madrid_project/Data/Processed/02_cleaning_audit"
)

AUDIT_DIR.mkdir(parents=True, exist_ok=True)

print("PHASE 4.2 — DATA QUALITY INSPECTION")
print("Datasets available:", len(datasets))
print("Output directory:", AUDIT_DIR)


PHASE 4.2 — DATA QUALITY INSPECTION
Datasets available: 10
Output directory: /content/real_madrid_project/Data/Processed/02_cleaning_audit


In [10]:

missing_reports = []

for name, df in datasets.items():

    for col in df.columns:

        missing_count = int(df[col].isna().sum())

        missing_reports.append({
            "dataset": name,
            "column": col,
            "data_type": str(df[col].dtype),
            "total_rows": len(df),
            "missing_values": missing_count,
            "missing_percentage": round(
                missing_count / len(df) * 100, 2
            ) if len(df) else 0,
            "unique_values": df[col].nunique()
        })

missing_df = pd.DataFrame(missing_reports)

missing_df = missing_df.sort_values(
    ["dataset", "missing_percentage"],
    ascending=[True, False]
)

display(missing_df.head(40))

missing_df.to_csv(
    AUDIT_DIR / "missing_values_audit.csv",
    index=False
)

print("Missing-value audit completed.")


,dataset,column,data_type,total_rows,missing_values,missing_percentage,unique_values
285,financials,segment,object,204,198,97.06,6
293,financials,source_pdf_page,float64,204,7,3.43,12
283,financials,reporting_period,object,204,0,0.00,11
284,financials,metric_name,object,204,0,0.00,30
286,financials,value_original,float64,204,0,0.00,201
287,financials,original_unit,object,204,0,0.00,2
288,financials,value_eur_million,float64,204,0,0.00,191
289,financials,currency,object,204,0,0.00,1
290,financials,data_type,object,204,0,0.00,3
291,financials,entity_scope,object,204,0,0.00,3


Missing-value audit completed.


In [11]:

KEYS = {
    "matches": ["match_id"],
    "match_team_stats": ["match_id"],
    "player_season_stats": ["player_id", "season"],
    "player_competition_stats": [
        "player_id", "season", "competition"
    ],
    "transfers": ["transfer_id"],
    "player_id_crosswalk": ["player_id"],
    "financials": [
        "reporting_period",
        "metric_name",
        "segment",
        "data_type",
        "entity_scope"
    ],
    "trophies": ["trophy_id"],
    "la_liga_standings": ["season"]
}

duplicate_reports = []

for name, df in datasets.items():

    exact_duplicates = int(
        df.duplicated().sum()
    )

    key = KEYS.get(name)

    if key and len(df):

        available_keys = [
            col for col in key if col in df.columns
        ]

        missing_key_values = int(
            df[available_keys].isna().any(axis=1).sum()
        )

        duplicate_key_rows = int(
            df.duplicated(
                subset=available_keys,
                keep=False
            ).sum()
        )

    else:
        missing_key_values = None
        duplicate_key_rows = None

    duplicate_reports.append({
        "dataset": name,
        "exact_duplicate_rows": exact_duplicates,
        "duplicate_key_rows": duplicate_key_rows,
        "rows_with_missing_key": missing_key_values
    })

duplicates_df = pd.DataFrame(duplicate_reports)

display(duplicates_df)

duplicates_df.to_csv(
    AUDIT_DIR / "duplicate_audit.csv",
    index=False
)

print("Duplicate audit completed.")


,dataset,exact_duplicate_rows,duplicate_key_rows,rows_with_missing_key
0,matches,0,0.0,0.0
1,match_team_stats,0,0.0,0.0
2,player_season_stats,0,0.0,0.0
3,player_competition_stats,0,0.0,0.0
4,player_match_stats,0,NaN,NaN
5,transfers,0,0.0,0.0
6,player_id_crosswalk,0,0.0,0.0
7,financials,0,0.0,198.0
8,trophies,0,0.0,0.0
9,la_liga_standings,0,0.0,0.0


Duplicate audit completed.


In [12]:

DATE_COLUMNS = {
    "matches": ["match_date"],
    "match_team_stats": ["match_date"],
    "transfers": ["transfer_date", "market_value_date"],
    "trophies": ["award_date"]
}

date_reports = []

for name, columns in DATE_COLUMNS.items():

    df = datasets[name]

    for col in columns:

        if col not in df.columns:
            continue

        values = df[col]

        parsed = pd.to_datetime(
            values,
            errors="coerce"
        )

        invalid = (
            values.notna() & parsed.isna()
        )

        date_reports.append({
            "dataset": name,
            "column": col,
            "total_rows": len(df),
            "valid_dates": int(parsed.notna().sum()),
            "missing_dates": int(values.isna().sum()),
            "invalid_dates": int(invalid.sum())
        })

date_df = pd.DataFrame(date_reports)

display(date_df)

date_df.to_csv(
    AUDIT_DIR / "date_format_audit.csv",
    index=False
)

print("Date-format audit completed.")


,dataset,column,total_rows,valid_dates,missing_dates,invalid_dates
0,matches,match_date,630,630,0,0
1,match_team_stats,match_date,630,630,0,0
2,transfers,transfer_date,214,0,214,0
3,transfers,market_value_date,214,0,214,0
4,trophies,award_date,22,18,4,0


Date-format audit completed.


In [13]:

print("PHASE 4.2 — INITIAL AUDIT SUMMARY")
print("=" * 50)

print("\nMISSING VALUES")
print(
    missing_df.groupby("dataset")["missing_values"]
    .sum()
    .to_string()
)

print("\nDUPLICATE RECORDS")
display(duplicates_df)

print("\nDATE VALIDATION")
display(date_df)

print("\nAUDIT REPORTS SAVED TO:")
print(AUDIT_DIR)


PHASE 4.2 — INITIAL AUDIT SUMMARY

MISSING VALUES
dataset
financials                    205
la_liga_standings               0
match_team_stats             2844
matches                      7834
player_competition_stats    20136
player_id_crosswalk             0
player_match_stats              0
player_season_stats           945
transfers                    1272
trophies                        4

DUPLICATE RECORDS


,dataset,exact_duplicate_rows,duplicate_key_rows,rows_with_missing_key
0,matches,0,0.0,0.0
1,match_team_stats,0,0.0,0.0
2,player_season_stats,0,0.0,0.0
3,player_competition_stats,0,0.0,0.0
4,player_match_stats,0,NaN,NaN
5,transfers,0,0.0,0.0
6,player_id_crosswalk,0,0.0,0.0
7,financials,0,0.0,198.0
8,trophies,0,0.0,0.0
9,la_liga_standings,0,0.0,0.0



DATE VALIDATION


,dataset,column,total_rows,valid_dates,missing_dates,invalid_dates
0,matches,match_date,630,630,0,0
1,match_team_stats,match_date,630,630,0,0
2,transfers,transfer_date,214,0,214,0
3,transfers,market_value_date,214,0,214,0
4,trophies,award_date,22,18,4,0



AUDIT REPORTS SAVED TO:
/content/real_madrid_project/Data/Processed/02_cleaning_audit


In [14]:

column_missing = []

for name, df in datasets.items():

    if df.empty:
        continue

    for col in df.columns:
        missing = int(df[col].isna().sum())

        if missing > 0:
            column_missing.append({
                "dataset": name,
                "column": col,
                "missing_count": missing,
                "missing_percentage": round(
                    missing / len(df) * 100, 2
                ),
                "available_values": int(df[col].notna().sum())
            })

column_missing_df = pd.DataFrame(column_missing)

column_missing_df = column_missing_df.sort_values(
    ["dataset", "missing_percentage"],
    ascending=[True, False]
)

display(column_missing_df)

column_missing_df.to_csv(
    AUDIT_DIR / "column_missingness_details.csv",
    index=False
)


,dataset,column,missing_count,missing_percentage,available_values
173,financials,segment,198,97.06,6
174,financials,source_pdf_page,7,3.43,197
42,match_team_stats,source_url,630,100.00,0
41,match_team_stats,stadium,612,97.14,18
31,match_team_stats,competition_edition,596,94.60,34
...,...,...,...,...,...
172,transfers,market_value_date,214,100.00,0
169,transfers,loan_fee,194,90.65,20
167,transfers,transfer_fee,111,51.87,103
168,transfers,fee_currency,111,51.87,103


In [15]:

financial_df = datasets["financials"].copy()

financial_dimensions = [
    "reporting_period",
    "metric_name",
    "segment",
    "data_type",
    "entity_scope"
]

available_dimensions = [
    c for c in financial_dimensions
    if c in financial_df.columns
]

print("FINANCIAL DATA STRUCTURE")
print("Columns:", financial_df.columns.tolist())

print("\nMissing values by dimension:")

display(
    financial_df[available_dimensions]
    .isna()
    .sum()
    .to_frame("missing_count")
)

print("\nFinancial rows with missing dimensions:")

financial_missing = financial_df[
    financial_df[available_dimensions].isna().any(axis=1)
]

display(financial_missing.head(30))

financial_missing.to_csv(
    AUDIT_DIR / "financial_missing_dimensions.csv",
    index=False
)


FINANCIAL DATA STRUCTURE
Columns: ['reporting_period', 'metric_name', 'segment', 'value_original', 'original_unit', 'value_eur_million', 'currency', 'data_type', 'entity_scope', 'source_document', 'source_pdf_page', 'source_note', 'source_page_reference', 'comparability_note']

Missing values by dimension:


,missing_count
reporting_period,0
metric_name,0
segment,198
data_type,0
entity_scope,0



Financial rows with missing dimensions:


,reporting_period,metric_name,segment,value_original,original_unit,value_eur_million,currency,data_type,entity_scope,source_document,source_pdf_page,source_note,source_page_reference,comparability_note
0,2016-17,stadium_membership_revenue,NaN,166569.0,EUR_thousand,166.569,EUR,Actual,Club standalone,Management Report Financial Statements 2016-20...,15.0,Consolidated/standalone statement line; curren...,PDF page 15,Standalone club reporting basis; not directly ...
1,2016-17,international_friendly_revenue,NaN,85858.0,EUR_thousand,85.858,EUR,Actual,Club standalone,Management Report Financial Statements 2016-20...,15.0,Consolidated/standalone statement line; curren...,PDF page 15,Standalone club reporting basis; not directly ...
2,2016-17,broadcasting_revenue,NaN,165299.0,EUR_thousand,165.299,EUR,Actual,Club standalone,Management Report Financial Statements 2016-20...,15.0,Consolidated/standalone statement line; curren...,PDF page 15,Standalone club reporting basis; not directly ...
3,2016-17,marketing_revenue,NaN,254138.0,EUR_thousand,254.138,EUR,Actual,Club standalone,Management Report Financial Statements 2016-20...,15.0,Consolidated/standalone statement line; curren...,PDF page 15,Standalone club reporting basis; not directly ...
4,2016-17,personnel_expenses,NaN,-406109.0,EUR_thousand,-406.109,EUR,Actual,Club standalone,Management Report Financial Statements 2016-20...,15.0,Consolidated/standalone statement line; curren...,PDF page 15,Standalone club reporting basis; not directly ...
5,2016-17,depreciation_amortization,NaN,-110157.0,EUR_thousand,-110.157,EUR,Actual,Club standalone,Management Report Financial Statements 2016-20...,15.0,Consolidated/standalone statement line; curren...,PDF page 15,Standalone club reporting basis; not directly ...
6,2016-17,profit_before_tax,NaN,26265.0,EUR_thousand,26.265,EUR,Actual,Club standalone,Management Report Financial Statements 2016-20...,15.0,Consolidated/standalone statement line; curren...,PDF page 15,Standalone club reporting basis; not directly ...
7,2016-17,profit_after_tax,NaN,21372.0,EUR_thousand,21.372,EUR,Actual,Club standalone,Management Report Financial Statements 2016-20...,15.0,Consolidated/standalone statement line; curren...,PDF page 15,Standalone club reporting basis; not directly ...
8,2016-17,other_operating_income,NaN,1030.0,EUR_thousand,1.030,EUR,Actual,Club standalone,Management Report Financial Statements 2016-20...,15.0,Statement line; current-year column.,PDF page 15,Standalone club reporting basis; not directly ...
9,2016-17,net_finance_income_expense,NaN,-1354.0,EUR_thousand,-1.354,EUR,Actual,Club standalone,Management Report Financial Statements 2016-20...,15.0,Statement line; current-year column.,PDF page 15,Standalone club reporting basis; not directly ...


In [16]:

trophy_df = datasets["trophies"].copy()

print("TROPHY DATASET")

display(trophy_df[
    trophy_df["award_date"].isna()
])

print("\nAll trophy records:")

display(trophy_df)


TROPHY DATASET


,trophy_id,season,season_role,competition,competition_edition,award_date,title_status,evidence_type,evidence_reference,source_name,source_file,notes
2,TROPHY-2016-17-la-liga,2016-17,Main,La Liga,2016-17,NaN,Won,League table calculated from supplied full-sea...,Real Madrid position 1; points 93; W-D-L 29-6-3,Football-Data.co.uk format; original source UR...,all da csvs.zip::la liga/LALIGA 2016-17.csv,Derived from full-season standings; validate a...
10,TROPHY-2019-20-la-liga,2019-20,Main,La Liga,2019-20,NaN,Won,League table calculated from supplied full-sea...,Real Madrid position 1; points 87; W-D-L 26-9-3,Football-Data.co.uk format; original source UR...,all da csvs.zip::la liga/LALIGA 2019-20.csv,Derived from full-season standings; validate a...
12,TROPHY-2021-22-la-liga,2021-22,Main,La Liga,2021-22,NaN,Won,League table calculated from supplied full-sea...,Real Madrid position 1; points 86; W-D-L 26-8-4,Football-Data.co.uk format; original source UR...,all da csvs.zip::la liga/LALIGA 2021-22.csv,Derived from full-season standings; validate a...
18,TROPHY-2023-24-la-liga,2023-24,Main,La Liga,2023-24,NaN,Won,League table calculated from supplied full-sea...,Real Madrid position 1; points 95; W-D-L 29-8-1,Football-Data.co.uk format; original source UR...,all da csvs.zip::la liga/LALIGA 2023-24.csv,Derived from full-season standings; validate a...



All trophy records:


,trophy_id,season,season_role,competition,competition_edition,award_date,title_status,evidence_type,evidence_reference,source_name,source_file,notes
0,TROPHY-2015-16-uefa-champions-league-2016-05-28,2015-16,Baseline,UEFA Champions League,2016,2016-05-28,Won,Decisive match result in supplied match log,match_id=RM-2015-16-uefa-champions-league-2016...,FBref / Sports Reference,all sportsref excel.zip::sportsref_download md...,Derived from decisive match result; cross-chec...
1,TROPHY-2016-17-fifa-club-world-cup-2016.0,2016-17,Main,FIFA Club World Cup,2016.0,2016-12-18,Won,Decisive match result in supplied match log,match_id=RM-2016-17-fifa-club-world-cup-2016-1...,OpenFootball,all da csvs.zip::fifa club world cup/2016/club...,Derived from decisive match result; cross-chec...
2,TROPHY-2016-17-la-liga,2016-17,Main,La Liga,2016-17,NaN,Won,League table calculated from supplied full-sea...,Real Madrid position 1; points 93; W-D-L 29-6-3,Football-Data.co.uk format; original source UR...,all da csvs.zip::la liga/LALIGA 2016-17.csv,Derived from full-season standings; validate a...
3,TROPHY-2016-17-uefa-champions-league-2017-06-03,2016-17,Main,UEFA Champions League,2017,2017-06-03,Won,Decisive match result in supplied match log,match_id=RM-2016-17-uefa-champions-league-2017...,FBref / Sports Reference,all sportsref excel.zip::sportsref_download md...,Derived from decisive match result; cross-chec...
4,TROPHY-2016-17-uefa-super-cup-2016.0,2016-17,Main,UEFA Super Cup,2016.0,2016-08-09,Won,Decisive match result in supplied match log,match_id=RM-2016-17-uefa-super-cup-2016-08-09-...,RSSSF supplied CSV; referee/attendance correct...,all da csvs.zip::uefa super cup/Uefa_Super_Cup...,Derived from decisive match result; cross-chec...
5,TROPHY-2017-18-fifa-club-world-cup-2017.0,2017-18,Main,FIFA Club World Cup,2017.0,2017-12-16,Won,Decisive match result in supplied match log,match_id=RM-2017-18-fifa-club-world-cup-2017-1...,OpenFootball,all da csvs.zip::fifa club world cup/2017/club...,Derived from decisive match result; cross-chec...
6,TROPHY-2017-18-spanish-super-cup-2017.0,2017-18,Main,Spanish Super Cup,2017.0,2017-08-16,Won,Decisive match result in supplied match log,match_id=RM-2017-18-spanish-super-cup-2017-08-...,FBref match log + supplied Spanish Super Cup CSV,all da csvs.zip::spanish supercup/Real_Madrid_...,Derived from decisive match result; cross-chec...
7,TROPHY-2017-18-uefa-champions-league-2018-05-26,2017-18,Main,UEFA Champions League,2018,2018-05-26,Won,Decisive match result in supplied match log,match_id=RM-2017-18-uefa-champions-league-2018...,FBref / Sports Reference,all sportsref excel.zip::sportsref_download md...,Derived from decisive match result; cross-chec...
8,TROPHY-2017-18-uefa-super-cup-2017.0,2017-18,Main,UEFA Super Cup,2017.0,2017-08-08,Won,Decisive match result in supplied match log,match_id=RM-2017-18-uefa-super-cup-2017-08-08-...,RSSSF supplied CSV; referee/attendance correct...,all da csvs.zip::uefa super cup/Uefa_Super_Cup...,Derived from decisive match result; cross-chec...
9,TROPHY-2018-19-fifa-club-world-cup-2018.0,2018-19,Main,FIFA Club World Cup,2018.0,2018-12-22,Won,Decisive match result in supplied match log,match_id=RM-2018-19-fifa-club-world-cup-2018-1...,OpenFootball,all da csvs.zip::fifa club world cup/2018/club...,Derived from decisive match result; cross-chec...


In [17]:

competition_df = datasets[
    "player_competition_stats"
].copy()

competition_missing = []

for col in competition_df.columns:

    missing = int(competition_df[col].isna().sum())

    if missing > 0:
        competition_missing.append({
            "column": col,
            "missing": missing,
            "percentage": round(
                missing / len(competition_df) * 100, 2
            ),
            "unique_values": competition_df[col].nunique()
        })

competition_missing_df = pd.DataFrame(
    competition_missing
).sort_values("percentage", ascending=False)

display(competition_missing_df.head(40))

competition_missing_df.to_csv(
    AUDIT_DIR / "player_competition_missingness.csv",
    index=False
)


,column,missing,percentage,unique_values
18,stats_keeper_569_Penalty_Kicks_Save,342,99.13,1
6,stats_keeper_569_Performance_SoTA,332,96.23,10
8,stats_keeper_569_Performance_Save,332,96.23,10
4,stats_keeper_569_Performance_GA,331,95.94,8
5,stats_keeper_569_Performance_GA90,331,95.94,12
7,stats_keeper_569_Performance_Saves,329,95.36,10
13,stats_keeper_569_Performance_CS__2,328,95.07,8
3,stats_keeper_569_Playing_Time_90s,327,94.78,11
0,stats_keeper_569_Playing_Time_MP,327,94.78,6
1,stats_keeper_569_Playing_Time_Starts,327,94.78,7


In [18]:

comp = datasets["player_competition_stats"].copy()

# Check the available identification and source fields
metadata = [
    "player_name",
    "player_id",
    "season",
    "competition",
    "position",
    "source_file",
    "source_name"
]

available = [c for c in metadata if c in comp.columns]

print("AVAILABLE IDENTIFICATION FIELDS")
display(comp[available].head(15))

# Coverage by season and competition
group_columns = [
    c for c in ["season", "competition"]
    if c in comp.columns
]

coverage = (
    comp.groupby(group_columns, dropna=False)
    .agg(
        records=("player_name", "size"),
        unique_players=("player_name", "nunique")
    )
    .reset_index()
)

print("\nCOVERAGE BY SEASON AND COMPETITION")
display(coverage)

# Source-file coverage
if "source_file" in comp.columns:
    print("\nSOURCE FILE COVERAGE")
    display(
        comp["source_file"]
        .fillna("MISSING SOURCE")
        .value_counts()
        .to_frame("records")
    )

# How many statistical fields are populated per player?
non_metric = [
    c for c in metadata
    if c in comp.columns
]

metric_columns = [
    c for c in comp.columns
    if c not in non_metric
]

comp["populated_metric_fields"] = (
    comp[metric_columns].notna().sum(axis=1)
)

print("\nSTATISTICS POPULATED PER RECORD")
display(
    comp["populated_metric_fields"]
    .describe()
    .to_frame("value")
)

display(
    comp[
        available + ["populated_metric_fields"]
    ].head(25)
)


AVAILABLE IDENTIFICATION FIELDS


,player_name,player_id,season,competition,position,source_file,source_name
0,Borja Mayoral,RMPLR-borja-mayoral,2015-16,Copa del Rey,FW,NaN,FBref
1,Casemiro,RMPLR-casemiro,2015-16,Copa del Rey,"DF,MF",NaN,FBref
2,Denis Cheryshev,RMPLR-denis-cheryshev,2015-16,Copa del Rey,"FW,MF",NaN,FBref
3,Isco,RMPLR-isco,2015-16,Copa del Rey,"FW,MF",NaN,FBref
4,James Rodríguez,RMPLR-james-rodriguez,2015-16,Copa del Rey,"FW,MF",NaN,FBref
5,Jesé,RMPLR-jese,2015-16,Copa del Rey,"FW,MF",NaN,FBref
6,Kiko Casilla,RMPLR-kiko-casilla,2015-16,Copa del Rey,GK,NaN,FBref
7,Lucas Vázquez,RMPLR-lucas-vazquez,2015-16,Copa del Rey,"DF,FW",NaN,FBref
8,Marcos Llorente,RMPLR-marcos-llorente,2015-16,Copa del Rey,"DF,MF",NaN,FBref
9,Mateo Kovačić,RMPLR-mateo-kovacic,2015-16,Copa del Rey,MF,NaN,FBref



COVERAGE BY SEASON AND COMPETITION


,season,competition,records,unique_players
0,2015-16,Copa del Rey,16,16
1,2016-17,Copa del Rey,27,27
2,2017-18,Copa del Rey,29,29
3,2018-19,Copa del Rey,34,34
4,2019-20,Copa del Rey,22,22
5,2020-21,Copa del Rey,19,19
6,2021-22,Copa del Rey,29,29
7,2022-23,Copa del Rey,32,32
8,2022-23,FIFA Club World Cup,23,23
9,2023-24,Copa del Rey,26,26



SOURCE FILE COVERAGE


,records
source_file,
MISSING SOURCE,292
sportsref_download 25.xls,30
sportsref_download 22.xls,23



STATISTICS POPULATED PER RECORD


,value
count,345.000000
mean,50.481159
std,21.511228
min,8.000000
25%,23.000000
50%,63.000000
75%,66.000000
max,85.000000


,player_name,player_id,season,competition,position,source_file,source_name,populated_metric_fields
0,Borja Mayoral,RMPLR-borja-mayoral,2015-16,Copa del Rey,FW,NaN,FBref,16
1,Casemiro,RMPLR-casemiro,2015-16,Copa del Rey,"DF,MF",NaN,FBref,52
2,Denis Cheryshev,RMPLR-denis-cheryshev,2015-16,Copa del Rey,"FW,MF",NaN,FBref,53
3,Isco,RMPLR-isco,2015-16,Copa del Rey,"FW,MF",NaN,FBref,52
4,James Rodríguez,RMPLR-james-rodriguez,2015-16,Copa del Rey,"FW,MF",NaN,FBref,53
5,Jesé,RMPLR-jese,2015-16,Copa del Rey,"FW,MF",NaN,FBref,52
6,Kiko Casilla,RMPLR-kiko-casilla,2015-16,Copa del Rey,GK,NaN,FBref,67
7,Lucas Vázquez,RMPLR-lucas-vazquez,2015-16,Copa del Rey,"DF,FW",NaN,FBref,52
8,Marcos Llorente,RMPLR-marcos-llorente,2015-16,Copa del Rey,"DF,MF",NaN,FBref,52
9,Mateo Kovačić,RMPLR-mateo-kovacic,2015-16,Copa del Rey,MF,NaN,FBref,53


In [19]:

fin = datasets["financials"].copy()

print("FINANCIAL DATA — MISSINGNESS BY COLUMN")

financial_missing = (
    fin.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_records")
)

financial_missing["missing_percentage"] = (
    financial_missing["missing_records"] /
    len(fin) * 100
).round(2)

display(financial_missing)

print("\nFINANCIAL DATA — REPORTING SCOPE")

for col in [
    "reporting_period",
    "metric_name",
    "segment",
    "data_type",
    "entity_scope"
]:
    if col in fin.columns:
        print(f"\n{col}")
        display(
            fin[col]
            .fillna("BLANK / NOT REPORTED")
            .value_counts()
            .head(20)
            .to_frame("records")
        )

print("\nROWS WITH MISSING REPORTING PERIOD OR METRIC")

required_financial = [
    c for c in ["reporting_period", "metric_name"]
    if c in fin.columns
]

display(
    fin[
        fin[required_financial].isna().any(axis=1)
    ]
)


FINANCIAL DATA — MISSINGNESS BY COLUMN


,missing_records,missing_percentage
segment,198,97.06
source_pdf_page,7,3.43
metric_name,0,0.00
value_original,0,0.00
original_unit,0,0.00
reporting_period,0,0.00
value_eur_million,0,0.00
currency,0,0.00
entity_scope,0,0.00
data_type,0,0.00



FINANCIAL DATA — REPORTING SCOPE

reporting_period


,records
reporting_period,
2025-26,36
2015-16,17
2016-17,17
2017-18,17
2018-19,17
2022-23,17
2019-20,17
2023-24,17
2024-25,17



metric_name


,records
metric_name,
operating_income_before_disposals,12
EBITDA,12
broadcasting_revenue,11
marketing_revenue,11
personnel_expenses,11
stadium_membership_revenue,11
depreciation_amortization,11
profit_before_tax,11
net_finance_income_expense,11


... (4 outputs left out)


entity_scope


,records
entity_scope,
Club standalone,101
Consolidated group,96
Club/Group summary; metric-specific perimeter,7



ROWS WITH MISSING REPORTING PERIOD OR METRIC


,reporting_period,metric_name,segment,value_original,original_unit,value_eur_million,currency,data_type,entity_scope,source_document,source_pdf_page,source_note,source_page_reference,comparability_note


In [20]:

comp = datasets["player_competition_stats"].copy()

# Identify actual FBref standard statistics
standard_cols = [
    c for c in comp.columns
    if c.startswith("standard_")
]

print("STANDARD STATISTIC COLUMNS:", len(standard_cols))

comp["standard_fields_available"] = (
    comp[standard_cols].notna().sum(axis=1)
)

comp["has_standard_stats"] = (
    comp["standard_fields_available"] > 0
)

# Coverage by competition, season and position
group_cols = [
    c for c in ["season", "competition", "position"]
    if c in comp.columns
]

coverage_audit = (
    comp.groupby(group_cols, dropna=False)
    .agg(
        player_records=("player_id", "size"),
        players=("player_id", "nunique"),
        records_with_standard_stats=(
            "has_standard_stats", "sum"
        ),
        average_standard_fields=(
            "standard_fields_available", "mean"
        )
    )
    .reset_index()
)

display(coverage_audit)

coverage_audit.to_csv(
    AUDIT_DIR / "standard_stats_coverage.csv",
    index=False
)

# Investigate missing source filenames
print("\nSOURCE FILE COVERAGE")

source_audit = (
    comp.assign(
        source_file=comp["source_file"].fillna(
            "MISSING SOURCE"
        )
    )
    .groupby(
        ["season", "competition", "source_file"],
        dropna=False
    )
    .size()
    .reset_index(name="records")
)

display(source_audit)

source_audit.to_csv(
    AUDIT_DIR / "source_provenance_audit.csv",
    index=False
)

# Inspect the player-season dataset
print("\nPLAYER SEASON DATASET")

season_df = datasets["player_season_stats"]

print("Columns:")
print(season_df.columns.tolist())

if "season" in season_df.columns:
    display(
        season_df.groupby("season")
        .size()
        .reset_index(name="player_records")
    )


STANDARD STATISTIC COLUMNS: 17


,season,competition,position,player_records,players,records_with_standard_stats,average_standard_fields
0,2015-16,Copa del Rey,DF,4,4,0,0.0
1,2015-16,Copa del Rey,"DF,FW",1,1,0,0.0
2,2015-16,Copa del Rey,"DF,MF",3,3,0,0.0
3,2015-16,Copa del Rey,FW,1,1,0,0.0
4,2015-16,Copa del Rey,"FW,MF",4,4,0,0.0
...,...,...,...,...,...,...,...
81,2025-26,Copa del Rey,FW,4,4,0,0.0
82,2025-26,Copa del Rey,"FW,MF",1,1,0,0.0
83,2025-26,Copa del Rey,GK,4,4,0,0.0
84,2025-26,Copa del Rey,MF,8,8,0,0.0



SOURCE FILE COVERAGE


,season,competition,source_file,records
0,2015-16,Copa del Rey,MISSING SOURCE,16
1,2016-17,Copa del Rey,MISSING SOURCE,27
2,2017-18,Copa del Rey,MISSING SOURCE,29
3,2018-19,Copa del Rey,MISSING SOURCE,34
4,2019-20,Copa del Rey,MISSING SOURCE,22
5,2020-21,Copa del Rey,MISSING SOURCE,19
6,2021-22,Copa del Rey,MISSING SOURCE,29
7,2022-23,Copa del Rey,MISSING SOURCE,32
8,2022-23,FIFA Club World Cup,sportsref_download 22.xls,23
9,2023-24,Copa del Rey,MISSING SOURCE,26



PLAYER SEASON DATASET
Columns:
['player_id', 'player_key_candidate', 'player_name', 'nation', 'position', 'age', 'season', 'season_role', 'competition_scope', 'appearances', 'starts', 'minutes', 'nineties', 'goals', 'assists', 'goal_contributions', 'non_penalty_goals', 'penalties_scored', 'penalties_attempted', 'yellow_cards', 'red_cards', 'goals_per90', 'assists_per90', 'goal_contributions_per90', 'non_penalty_goals_per90', 'goal_contributions_minus_penalties_per90', 'source_name', 'source_file', 'source_row']


,season,player_records
0,2015-16,29
1,2016-17,28
2,2017-18,32
3,2018-19,36
4,2019-20,32
5,2020-21,37
6,2021-22,36
7,2022-23,36
8,2023-24,38
9,2024-25,38


In [21]:

fin = datasets["financials"].copy()

missing_pages = fin[
    fin["source_pdf_page"].isna()
]

print("FINANCIAL RECORDS WITHOUT PDF PAGE NUMBERS")

columns_to_show = [
    "reporting_period",
    "metric_name",
    "entity_scope",
    "data_type",
    "source_document",
    "source_page_reference",
    "source_note"
]

columns_to_show = [
    c for c in columns_to_show if c in fin.columns
]

display(missing_pages[columns_to_show])

missing_pages.to_csv(
    AUDIT_DIR / "financial_page_reference_gaps.csv",
    index=False
)

print("Financial documentation audit completed.")


FINANCIAL RECORDS WITHOUT PDF PAGE NUMBERS


,reporting_period,metric_name,entity_scope,data_type,source_document,source_page_reference,source_note
197,2025-26,operating_income_before_disposals,Club/Group summary; metric-specific perimeter,Actual announcement,REAL MADRID CLOSES THE 2025-2026 SEASON WITH T...,HTML economic-financial summary table,Official 2025/26 economic-financial summary ta...
198,2025-26,EBITDA_before_disposals,Club/Group summary; metric-specific perimeter,Actual announcement,REAL MADRID CLOSES THE 2025-2026 SEASON WITH T...,HTML economic-financial summary table,Official 2025/26 economic-financial summary ta...
199,2025-26,EBITDA,Club/Group summary; metric-specific perimeter,Actual announcement,REAL MADRID CLOSES THE 2025-2026 SEASON WITH T...,HTML economic-financial summary table,Official 2025/26 economic-financial summary ta...
200,2025-26,net_profit_after_tax,Club/Group summary; metric-specific perimeter,Actual announcement,REAL MADRID CLOSES THE 2025-2026 SEASON WITH T...,HTML economic-financial summary table,Official 2025/26 economic-financial summary ta...
201,2025-26,net_equity,Club/Group summary; metric-specific perimeter,Actual announcement,REAL MADRID CLOSES THE 2025-2026 SEASON WITH T...,HTML economic-financial summary table,Official 2025/26 economic-financial summary ta...
202,2025-26,cash_and_cash_equivalents,Club/Group summary; metric-specific perimeter,Actual announcement,REAL MADRID CLOSES THE 2025-2026 SEASON WITH T...,HTML economic-financial summary table,Official 2025/26 economic-financial summary ta...
203,2025-26,net_debt_excluding_stadium_renovation,Club/Group summary; metric-specific perimeter,Actual announcement,REAL MADRID CLOSES THE 2025-2026 SEASON WITH T...,HTML economic-financial summary table,Official 2025/26 economic-financial summary ta...


Financial documentation audit completed.


In [22]:

import pandas as pd
import numpy as np

comp = datasets["player_competition_stats"].copy()
season = datasets["player_season_stats"].copy()

print("1. PLAYER COMPETITION COVERAGE")
display(
    comp.groupby("competition")
    .agg(
        records=("player_id", "size"),
        unique_players=("player_id", "nunique"),
        seasons=("season", "nunique")
    )
    .reset_index()
)

print("\n2. PLAYER SEASON COMPETITION SCOPE")

display(
    season.groupby(
        ["season", "competition_scope"],
        dropna=False
    )
    .agg(
        records=("player_id", "size"),
        unique_players=("player_id", "nunique")
    )
    .reset_index()
)

print("\n3. TOTAL COMPETITION SCOPE VALUES")

display(
    season["competition_scope"]
    .value_counts(dropna=False)
    .to_frame("records")
)

print("\n4. PLAYER SEASON SOURCE FILES")

display(
    season["source_file"]
    .fillna("MISSING SOURCE")
    .value_counts()
    .to_frame("records")
)


1. PLAYER COMPETITION COVERAGE


,competition,records,unique_players,seasons
0,Copa del Rey,292,111,11
1,FIFA Club World Cup,53,41,2



2. PLAYER SEASON COMPETITION SCOPE


,season,competition_scope,records,unique_players
0,2015-16,All competitions,29,29
1,2016-17,All competitions,28,28
2,2017-18,All competitions,32,32
3,2018-19,All competitions,36,36
4,2019-20,All competitions,32,32
5,2020-21,All competitions,37,37
6,2021-22,All competitions,36,36
7,2022-23,All competitions,36,36
8,2023-24,All competitions,38,38
9,2024-25,All competitions,38,38



3. TOTAL COMPETITION SCOPE VALUES


,records
competition_scope,
All competitions,386



4. PLAYER SEASON SOURCE FILES


,records
source_file,
all sportsref excel.zip::sportsref_download stats 25-26.xls,44
all sportsref excel.zip::sportsref_download stats 24-25.xls,38
all sportsref excel.zip::sportsref_download stats 23-24.xls,38
all sportsref excel.zip::sportsref_download stats 20-21.xls,37
sportsref_download.xls (corrected 2022-23),36
all sportsref excel.zip::sportsref_download stats 18-19.xls,36
all sportsref excel.zip::sportsref_download stats 21-22.xls,36
all sportsref excel.zip::sportsref_download stats 17-18.xls,32
all sportsref excel.zip::sportsref_download stats 19-20.xls,32


In [23]:

standard_cols = [
    c for c in comp.columns
    if c.startswith("standard_")
]

comp["standard_fields_populated"] = (
    comp[standard_cols].notna().sum(axis=1)
)

print("STANDARD STATISTICS COVERAGE BY COMPETITION")

standard_summary = (
    comp.groupby("competition")
    .agg(
        total_records=("player_id", "size"),
        records_with_standard_stats=(
            "standard_fields_populated",
            lambda x: (x > 0).sum()
        ),
        average_fields_populated=(
            "standard_fields_populated", "mean"
        ),
        maximum_fields_populated=(
            "standard_fields_populated", "max"
        )
    )
    .reset_index()
)

display(standard_summary)

print("\nSTANDARD STATISTICS COVERAGE BY SEASON")

display(
    comp.groupby(
        ["season", "competition"]
    )
    .agg(
        total_records=("player_id", "size"),
        records_with_stats=(
            "standard_fields_populated",
            lambda x: (x > 0).sum()
        )
    )
    .reset_index()
)

print("\nINDIVIDUAL STANDARD STATISTIC COVERAGE")

display(
    comp[standard_cols]
    .notna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("available_records")
)


STANDARD STATISTICS COVERAGE BY COMPETITION


,competition,total_records,records_with_standard_stats,average_fields_populated,maximum_fields_populated
0,Copa del Rey,292,0,0.000000,0
1,FIFA Club World Cup,53,53,13.603774,17



STANDARD STATISTICS COVERAGE BY SEASON


,season,competition,total_records,records_with_stats
0,2015-16,Copa del Rey,16,0
1,2016-17,Copa del Rey,27,0
2,2017-18,Copa del Rey,29,0
3,2018-19,Copa del Rey,34,0
4,2019-20,Copa del Rey,22,0
5,2020-21,Copa del Rey,19,0
6,2021-22,Copa del Rey,29,0
7,2022-23,Copa del Rey,32,0
8,2022-23,FIFA Club World Cup,23,23
9,2023-24,Copa del Rey,26,0



INDIVIDUAL STANDARD STATISTIC COVERAGE


,available_records
standard_MP,53
standard_Playing_Time_Starts,53
standard_Playing_Time_Min,41
standard_Playing_Time_90s,41
standard_Performance_Gls,41
standard_Performance_Ast,41
standard_Performance_G_A,41
standard_Performance_G_PK,41
standard_Performance_PK,41
standard_Performance_PKatt,41


In [24]:

from pathlib import Path

search_roots = [
    Path("/content/real_madrid_project")
]

extensions = [".xls", ".xlsx", ".csv"]

found_files = []

for root in search_roots:
    if root.exists():
        for path in root.rglob("*"):
            if path.is_file() and path.suffix.lower() in extensions:
                found_files.append({
                    "filename": path.name,
                    "extension": path.suffix.lower(),
                    "location": str(path)
                })

source_inventory = pd.DataFrame(found_files)

print("ORIGINAL DATA FILE INVENTORY")
print("Files discovered:", len(source_inventory))

display(source_inventory.head(100))

source_inventory.to_csv(
    AUDIT_DIR / "original_source_inventory.csv",
    index=False
)


ORIGINAL DATA FILE INVENTORY
Files discovered: 23


,filename,extension,location
0,Data_Quality_Log.csv,.csv,/content/real_madrid_project/Data/Cleaned/Real...
1,Real_Madrid_Player_ID_Crosswalk.csv,.csv,/content/real_madrid_project/Data/Cleaned/Real...
2,Source_Register.csv,.csv,/content/real_madrid_project/Data/Cleaned/Real...
3,Real_Madrid_Player_Competition_Stats.csv,.csv,/content/real_madrid_project/Data/Cleaned/Real...
4,Real_Madrid_Transfers.csv,.csv,/content/real_madrid_project/Data/Cleaned/Real...
5,Real_Madrid_Trophies.csv,.csv,/content/real_madrid_project/Data/Cleaned/Real...
6,Real_Madrid_Match_Team_Stats.csv,.csv,/content/real_madrid_project/Data/Cleaned/Real...
7,Real_Madrid_Player_Match_Stats.csv,.csv,/content/real_madrid_project/Data/Cleaned/Real...
8,Dataset_Inventory.csv,.csv,/content/real_madrid_project/Data/Cleaned/Real...
9,Real_Madrid_Player_Season_Stats.csv,.csv,/content/real_madrid_project/Data/Cleaned/Real...


In [25]:

import zipfile
import pandas as pd
from google.colab import files

uploaded_raw = files.upload()

raw_zip_path = next(
    name for name in uploaded_raw
    if name.lower().endswith(".zip")
)

with zipfile.ZipFile(raw_zip_path, "r") as archive:

    workbook_files = [
        name for name in archive.namelist()
        if name.lower().endswith((".xls", ".xlsx"))
    ]

    workbook_inventory = pd.DataFrame({
        "workbook": workbook_files
    })

print("ORIGINAL EXCEL WORKBOOK AUDIT")
print("Total workbooks:", len(workbook_inventory))

display(workbook_inventory)


Saving Real_Madrid_Master_Datasets_2026-11-18_UPDATED.zip to Real_Madrid_Master_Datasets_2026-11-18_UPDATED (1).zip
ORIGINAL EXCEL WORKBOOK AUDIT
Total workbooks: 0


,workbook


In [26]:

import zipfile
import pandas as pd
from google.colab import files
from pathlib import Path
import io

uploaded = files.upload()

archive_name = next(
    name for name in uploaded
    if name.lower().endswith(".zip")
)

def inspect_zip(data, archive_label):

    rows = []

    with zipfile.ZipFile(io.BytesIO(data)) as z:

        for name in z.namelist():

            if name.endswith("/"):
                continue

            extension = Path(name).suffix.lower()

            if extension in [".xls", ".xlsx", ".csv"]:
                rows.append({
                    "archive": archive_label,
                    "filename": name,
                    "extension": extension,
                    "size_bytes": z.getinfo(name).file_size
                })

            elif extension == ".zip":
                nested_data = z.read(name)
                rows.extend(
                    inspect_zip(nested_data, name)
                )

    return rows


inventory = pd.DataFrame(
    inspect_zip(
        uploaded[archive_name],
        archive_name
    )
)

print("ORIGINAL FBREF SOURCE INVENTORY")
print("Files discovered:", len(inventory))

display(inventory)

inventory.to_csv(
    AUDIT_DIR / "original_fbref_archive_inventory.csv",
    index=False
)


Saving Real_Madrid_Master_Datasets_2026-11-18_UPDATED.zip to Real_Madrid_Master_Datasets_2026-11-18_UPDATED (2).zip
ORIGINAL FBREF SOURCE INVENTORY
Files discovered: 13


,archive,filename,extension,size_bytes
0,Real_Madrid_Master_Datasets_2026-11-18_UPDATED...,Real_Madrid_Master_Datasets_2026-11-18_UPDATED...,.csv,10035
1,Real_Madrid_Master_Datasets_2026-11-18_UPDATED...,Real_Madrid_Master_Datasets_2026-11-18_UPDATED...,.csv,2426
2,Real_Madrid_Master_Datasets_2026-11-18_UPDATED...,Real_Madrid_Master_Datasets_2026-11-18_UPDATED...,.csv,242132
3,Real_Madrid_Master_Datasets_2026-11-18_UPDATED...,Real_Madrid_Master_Datasets_2026-11-18_UPDATED...,.csv,58250
4,Real_Madrid_Master_Datasets_2026-11-18_UPDATED...,Real_Madrid_Master_Datasets_2026-11-18_UPDATED...,.csv,2872
5,Real_Madrid_Master_Datasets_2026-11-18_UPDATED...,Real_Madrid_Master_Datasets_2026-11-18_UPDATED...,.csv,237636
6,Real_Madrid_Master_Datasets_2026-11-18_UPDATED...,Real_Madrid_Master_Datasets_2026-11-18_UPDATED...,.csv,329737
7,Real_Madrid_Master_Datasets_2026-11-18_UPDATED...,Real_Madrid_Master_Datasets_2026-11-18_UPDATED...,.csv,41435
8,Real_Madrid_Master_Datasets_2026-11-18_UPDATED...,Real_Madrid_Master_Datasets_2026-11-18_UPDATED...,.csv,299
9,Real_Madrid_Master_Datasets_2026-11-18_UPDATED...,Real_Madrid_Master_Datasets_2026-11-18_UPDATED...,.csv,92905


In [27]:

from google.colab import files
import zipfile
import io
import pandas as pd
from pathlib import Path

uploaded = files.upload()

for filename, content in uploaded.items():

    if not filename.lower().endswith(".zip"):
        continue

    print("\nARCHIVE SELECTED:", filename)

    with zipfile.ZipFile(io.BytesIO(content)) as archive:

        excel_files = []

        for item in archive.namelist():

            if item.lower().endswith((".xls", ".xlsx")):
                excel_files.append(item)

            elif item.lower().endswith(".zip"):

                nested = archive.read(item)

                with zipfile.ZipFile(
                    io.BytesIO(nested)
                ) as nested_archive:

                    for nested_item in nested_archive.namelist():

                        if nested_item.lower().endswith(
                            (".xls", ".xlsx")
                        ):
                            excel_files.append(
                                item + "::" + nested_item
                            )

        print("Excel workbooks found:", len(excel_files))

        for workbook in excel_files:
            print(workbook)


Saving all sportsref excel.zip to all sportsref excel.zip

ARCHIVE SELECTED: all sportsref excel.zip
Excel workbooks found: 22
sportsref_download md 21-22.xls
sportsref_download stats 22-23.xls
sportsref_download md 22-23.xls
sportsref_download stats 23-24.xls
sportsref_download md 23-24.xls
sportsref_download stats 24-25.xls
sportsref_download md 24-25.xls
sportsref_download stats 25-26.xls
sportsref_download md 25-26.xls
sportsref_download stats 15-16.xls
sportsref_download md 15-16.xls
sportsref_download stats 16-17.xls
sportsref_download md 16-17.xls
sportsref_download stats 17-18.xls
sportsref_download md 17-18.xls
sportsref_download stats 18-19.xls
sportsref_download md 18-19.xls
sportsref_download stats 19-20.xls
sportsref_download md 19-20.xls
sportsref_download stats 20-21.xls
sportsref_download md 20-21.xls
sportsref_download stats 21-22.xls


In [29]:

!pip -q install lxml beautifulsoup4 xlrd

import pandas as pd
import zipfile
import io
import warnings
from pathlib import Path

ZIP_PATH = Path("/content/all sportsref excel.zip")

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        "Upload all sportsref excel.zip to Colab first."
    )

AUDIT_DIR = Path(
    "/content/real_madrid_project/Data/Processed/02_cleaning_audit"
)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

workbook_reports = []
table_reports = []
fbref_tables = {}

with zipfile.ZipFile(ZIP_PATH, "r") as archive:

    excel_files = [
        name for name in archive.namelist()
        if name.lower().endswith((".xls", ".xlsx"))
    ]

    print("FBREF SOURCE INSPECTION")
    print("Workbooks found:", len(excel_files))

    for filename in excel_files:

        content = archive.read(filename)

        # Detect actual file format
        sample = content.lstrip()[:1000].lower()

        is_html = (
            b"<html" in sample
            or b"<!doctype html" in sample
            or b"<table" in sample
        )

        tables = []

        try:

            if is_html:

                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")

                    tables = pd.read_html(
                        io.BytesIO(content),
                        displayed_only=False
                    )

                file_format = "HTML disguised as XLS"

            else:

                tables = pd.read_excel(
                    io.BytesIO(content),
                    sheet_name=None,
                    engine="xlrd"
                )

                if isinstance(tables, dict):
                    tables = list(tables.values())

                file_format = "Excel workbook"

            workbook_type = (
                "stats" if "stats" in filename.lower()
                else "md"
            )

            workbook_reports.append({
                "workbook": filename,
                "workbook_type": workbook_type,
                "detected_format": file_format,
                "tables_found": len(tables),
                "status": "SUCCESS"
            })

            print(
                f"\nSUCCESS: {filename} "
                f"— {len(tables)} tables"
            )

            for i, df in enumerate(tables):

                # Preserve the original column labels for inspection
                headers = [
                    " | ".join(map(str, c))
                    if isinstance(c, tuple)
                    else str(c)
                    for c in df.columns
                ]

                table_key = f"{filename}::table_{i}"

                fbref_tables[table_key] = df.copy()

                table_reports.append({
                    "workbook": filename,
                    "workbook_type": workbook_type,
                    "table_number": i,
                    "rows": len(df),
                    "columns": len(df.columns),
                    "column_headers": " || ".join(headers)
                })

        except Exception as e:

            workbook_reports.append({
                "workbook": filename,
                "workbook_type": "unknown",
                "detected_format": (
                    "HTML" if is_html else "Unknown"
                ),
                "tables_found": 0,
                "status": f"ERROR: {str(e)[:250]}"
            })

            print(f"\nERROR: {filename}: {e}")

workbook_df = pd.DataFrame(workbook_reports)
table_df = pd.DataFrame(table_reports)

print("\n" + "=" * 60)
print("WORKBOOK INSPECTION SUMMARY")
display(workbook_df)

print("\n" + "=" * 60)
print("TABLE INVENTORY")
display(table_df)

# Save audit reports
workbook_df.to_csv(
    AUDIT_DIR / "fbref_workbook_inventory.csv",
    index=False
)

table_df.to_csv(
    AUDIT_DIR / "fbref_table_inventory.csv",
    index=False
)

print("\nTOTAL WORKBOOKS:", len(workbook_df))
print("SUCCESSFUL:", (workbook_df["status"] == "SUCCESS").sum())
print("TOTAL TABLES:", len(table_df))

print("\nAudit reports saved to:", AUDIT_DIR)


FBREF SOURCE INSPECTION
Workbooks found: 22

SUCCESS: sportsref_download md 21-22.xls — 1 tables

SUCCESS: sportsref_download stats 22-23.xls — 1 tables

SUCCESS: sportsref_download md 22-23.xls — 1 tables

SUCCESS: sportsref_download stats 23-24.xls — 1 tables

SUCCESS: sportsref_download md 23-24.xls — 1 tables

SUCCESS: sportsref_download stats 24-25.xls — 1 tables

SUCCESS: sportsref_download md 24-25.xls — 1 tables

SUCCESS: sportsref_download stats 25-26.xls — 1 tables

SUCCESS: sportsref_download md 25-26.xls — 1 tables

SUCCESS: sportsref_download stats 15-16.xls — 1 tables

SUCCESS: sportsref_download md 15-16.xls — 1 tables

SUCCESS: sportsref_download stats 16-17.xls — 1 tables

SUCCESS: sportsref_download md 16-17.xls — 1 tables

SUCCESS: sportsref_download stats 17-18.xls — 1 tables

SUCCESS: sportsref_download md 17-18.xls — 1 tables

SUCCESS: sportsref_download stats 18-19.xls — 1 tables

SUCCESS: sportsref_download md 18-19.xls — 1 tables

SUCCESS: sportsref_download st

,workbook,workbook_type,detected_format,tables_found,status
0,sportsref_download md 21-22.xls,md,HTML disguised as XLS,1,SUCCESS
1,sportsref_download stats 22-23.xls,stats,HTML disguised as XLS,1,SUCCESS
2,sportsref_download md 22-23.xls,md,HTML disguised as XLS,1,SUCCESS
3,sportsref_download stats 23-24.xls,stats,HTML disguised as XLS,1,SUCCESS
4,sportsref_download md 23-24.xls,md,HTML disguised as XLS,1,SUCCESS
5,sportsref_download stats 24-25.xls,stats,HTML disguised as XLS,1,SUCCESS
6,sportsref_download md 24-25.xls,md,HTML disguised as XLS,1,SUCCESS
7,sportsref_download stats 25-26.xls,stats,HTML disguised as XLS,1,SUCCESS
8,sportsref_download md 25-26.xls,md,HTML disguised as XLS,1,SUCCESS
9,sportsref_download stats 15-16.xls,stats,HTML disguised as XLS,1,SUCCESS



TABLE INVENTORY


,workbook,workbook_type,table_number,rows,columns,column_headers
0,sportsref_download md 21-22.xls,md,0,56,18,Date || Time || Comp || Round || Day || Venue ...
1,sportsref_download stats 22-23.xls,stats,0,25,22,Unnamed: 0_level_0 | Player || Unnamed: 1_leve...
2,sportsref_download md 22-23.xls,md,0,59,18,Date || Time || Comp || Round || Day || Venue ...
3,sportsref_download stats 23-24.xls,stats,0,38,22,Unnamed: 0_level_0 | Player || Unnamed: 1_leve...
4,sportsref_download md 23-24.xls,md,0,55,18,Date || Time || Comp || Round || Day || Venue ...
5,sportsref_download stats 24-25.xls,stats,0,38,22,Unnamed: 0_level_0 | Player || Unnamed: 1_leve...
6,sportsref_download md 24-25.xls,md,0,61,18,Date || Time || Comp || Round || Day || Venue ...
7,sportsref_download stats 25-26.xls,stats,0,44,22,Unnamed: 0_level_0 | Player || Unnamed: 1_leve...
8,sportsref_download md 25-26.xls,md,0,56,18,Date || Time || Comp || Round || Day || Venue ...
9,sportsref_download stats 15-16.xls,stats,0,29,22,Unnamed: 0_level_0 | Player || Unnamed: 1_leve...



TOTAL WORKBOOKS: 22
SUCCESSFUL: 22
TOTAL TABLES: 22

Audit reports saved to: /content/real_madrid_project/Data/Processed/02_cleaning_audit


In [30]:

# Display the first five rows of each extracted table

for key, df in fbref_tables.items():

    print("\n" + "=" * 75)
    print("SOURCE:", key)
    print("ROWS:", len(df))
    print("COLUMNS:", len(df.columns))

    display(df.head(5))



SOURCE: sportsref_download md 21-22.xls::table_0
ROWS: 56
COLUMNS: 18


,Date,Time,Comp,Round,Day,Venue,Result,GF,GA,Opponent,Poss,Attendance,Captain,Formation,Opp Formation,Referee,Match Report,Notes
0,2021-08-14,22:00 (02:00),La Liga,Matchweek 1,Sat,Away,W,4,1,AlavÃ©s,59,3968.0,Karim Benzema,4-3-3,4-2-3-1,CÃ©sar Soto,Match Report,NaN
1,2021-08-22,22:00 (02:00),La Liga,Matchweek 2,Sun,Away,D,3,3,Levante,61,9838.0,Karim Benzema,4-3-3,4-3-3,Guillermo Cuadra,Match Report,NaN
2,2021-08-28,22:00 (02:00),La Liga,Matchweek 3,Sat,Away,W,1,0,Real Betis,49,22590.0,Karim Benzema,4-3-3,4-2-3-1,Alejandro HernÃ¡ndez,Match Report,NaN
3,2021-09-12,21:00 (01:00),La Liga,Matchweek 4,Sun,Home,W,5,2,Celta Vigo,60,19874.0,Karim Benzema,4-4-2,4-1-3-2,JosÃ© SÃ¡nchez,Match Report,NaN
4,2021-09-15,21:00 (01:00),Champions Lg,Group stage,Wed,Away,W,1,0,it Inter,53,37082.0,Karim Benzema,4-3-3,3-5-2,Daniel Siebert,Match Report,NaN



SOURCE: sportsref_download stats 22-23.xls::table_0
ROWS: 25
COLUMNS: 22


Unnamed: 0_level_0 Unnamed: 1_level_0 Unnamed: 2_level_0 Unnamed: 3_level_0  \
              Player             Nation                Pos                Age   
0      Karim Benzema             fr FRA                 FW               34.0   
1   Thibaut Courtois             be BEL                 GK               30.0   
2   Vinicius JÃºnior             br BRA                 FW               22.0   
3   Antonio RÃ¼diger             de GER                 DF               29.0   
4  Federico Valverde             uy URU              MF,FW               24.0   

  Unnamed: 4_level_0 Playing Time             Performance       ...       \
                  MP       Starts    Min  90s         Gls  Ast  ...   PK   
0                  2            2  210.0  2.3         2.0  0.0  ...  1.0   
1                  2            2  210.0  2.3         0.0  0.0  ...  0.0   
2                  2            2  210.0  2.3         0.0  0.0  ...  0.0   
3                  2            2  210.0  2.3         0.0  0.0  ...  0.0   
4                  2            2  194.0  2.2         0.0  0.0  ...  0.0   

                  Per 90 Minutes                         Unnamed: 21_level_0  
  PKatt CrdY CrdR            Gls  Ast   G+A  G-PK G+A-PK             Matches  
0   1.0  0.0  0.0           0.86  0.0  0.86  0.43   0.43             Matches  
1   0.0  0.0  0.0           0.00  0.0  0.00  0.00   0.00             Matches  
2   0.0  0.0  0.0           0.00  0.0  0.00  0.00   0.00             Matches  
3   0.0  0.0  0.0           0.00  0.0  0.00  0.00   0.00             Matches  
4   0.0  1.0  0.0           0.00  0.0  0.00  0.00   0.00             Matches  

[5 rows x 22 columns]


SOURCE: sportsref_download md 22-23.xls::table_0
ROWS: 59
COLUMNS: 18


,Date,Time,Comp,Round,Day,Venue,Result,GF,GA,Opponent,Poss,Attendance,Captain,Formation,Opp Formation,Referee,Match Report,Notes
0,2022-08-10,22:00 (01:00),Super Cup,UEFA Super Cup,Wed,Home,W,2,0,de Frankfurt,58,31042,Karim Benzema,4-3-3,3-4-3,Michael Oliver,Match Report,NaN
1,2022-08-14,22:00 (02:00),La Liga,Matchweek 1,Sun,Away,W,2,1,AlmerÃ­a,68,14386,Karim Benzema,4-3-3,5-3-2,Juan MartÃ­nez,Match Report,NaN
2,2022-08-20,22:00 (02:00),La Liga,Matchweek 2,Sat,Away,W,4,1,Celta Vigo,49,15681,Karim Benzema,4-3-3,4-1-3-2,JesÃºs Gil,Match Report,NaN
3,2022-08-28,22:00 (02:00),La Liga,Matchweek 3,Sun,Away,W,3,1,Espanyol,64,25778,Karim Benzema,4-3-3,4-1-4-1,Mario Melero,Match Report,NaN
4,2022-09-03,16:15 (20:15),La Liga,Matchweek 4,Sat,Home,W,2,1,Real Betis,50,58579,Karim Benzema,4-3-3,4-2-3-1,JosÃ© SÃ¡nchez,Match Report,NaN


... (34 outputs left out)


SOURCE: sportsref_download md 20-21.xls::table_0
ROWS: 52
COLUMNS: 18


,Date,Time,Comp,Round,Day,Venue,Result,GF,GA,Opponent,Poss,Attendance,Captain,Formation,Opp Formation,Referee,Match Report,Notes
0,2020-09-20,21:00 (01:00),La Liga,Matchweek 2,Sun,Away,D,0,0,Real Sociedad,68,NaN,Sergio Ramos,4-2-3-1,4-4-2,Juan MartÃ­nez,Match Report,NaN
1,2020-09-26,21:00 (01:00),La Liga,Matchweek 3,Sat,Away,W,3,2,Real Betis,54,NaN,Sergio Ramos,4-3-1-2,4-2-3-1,Ricardo de Burgos,Match Report,NaN
2,2020-09-30,21:30 (01:30),La Liga,Matchweek 4,Wed,Home,W,1,0,Valladolid,63,NaN,Sergio Ramos,4-3-1-2,4-2-3-1,CÃ©sar Soto,Match Report,NaN
3,2020-10-04,16:00 (20:00),La Liga,Matchweek 5,Sun,Away,W,2,0,Levante,55,NaN,Sergio Ramos,4-3-3,4-4-2,JosÃ© Luis Munuera,Match Report,NaN
4,2020-10-17,18:30 (22:30),La Liga,Matchweek 6,Sat,Home,L,0,1,CÃ¡diz,75,NaN,Sergio Ramos,4-3-3,4-4-2,Santiago Jaime,Match Report,NaN



SOURCE: sportsref_download stats 21-22.xls::table_0
ROWS: 36
COLUMNS: 22


Unnamed: 0_level_0 Unnamed: 1_level_0 Unnamed: 2_level_0 Unnamed: 3_level_0  \
              Player             Nation                Pos                Age   
0   Thibaut Courtois             be BEL                 GK                 29   
1     Ãder MilitÃ£o             br BRA                 DF                 23   
2   Vinicius JÃºnior             br BRA                 FW                 21   
3        David Alaba             at AUT                 DF                 29   
4           Casemiro             br BRA                 MF                 29   

  Unnamed: 4_level_0 Playing Time               Performance        ...       \
                  MP       Starts     Min   90s         Gls   Ast  ...   PK   
0                 52           52  4770.0  53.0         0.0   0.0  ...  0.0   
1                 50           50  4491.0  49.9         2.0   2.0  ...  0.0   
2                 52           47  4253.0  47.3        22.0  16.0  ...  0.0   
3                 46           46  4066.0  45.2         3.0   4.0  ...  0.0   
4                 48           46  3918.0  43.5         1.0   3.0  ...  0.0   

                   Per 90 Minutes                          Unnamed: 21_level_0  
  PKatt  CrdY CrdR            Gls   Ast   G+A  G-PK G+A-PK             Matches  
0   0.0   1.0  0.0           0.00  0.00  0.00  0.00   0.00             Matches  
1   0.0   9.0  1.0           0.04  0.04  0.08  0.04   0.08             Matches  
2   0.0   7.0  0.0           0.47  0.34  0.80  0.47   0.80             Matches  
3   0.0   4.0  0.0           0.07  0.09  0.15  0.07   0.15             Matches  
4   0.0  16.0  0.0           0.02  0.07  0.09  0.02   0.09             Matches  

[5 rows x 22 columns]

In [31]:
# Display the first five rows of each extracted table

for key, df in fbref_tables.items():

    print("\n" + "=" * 75)
    print("SOURCE:", key)
    print("ROWS:", len(df))
    print("COLUMNS:", len(df.columns))

    display(df.head(5))



SOURCE: sportsref_download md 21-22.xls::table_0
ROWS: 56
COLUMNS: 18


,Date,Time,Comp,Round,Day,Venue,Result,GF,GA,Opponent,Poss,Attendance,Captain,Formation,Opp Formation,Referee,Match Report,Notes
0,2021-08-14,22:00 (02:00),La Liga,Matchweek 1,Sat,Away,W,4,1,AlavÃ©s,59,3968.0,Karim Benzema,4-3-3,4-2-3-1,CÃ©sar Soto,Match Report,NaN
1,2021-08-22,22:00 (02:00),La Liga,Matchweek 2,Sun,Away,D,3,3,Levante,61,9838.0,Karim Benzema,4-3-3,4-3-3,Guillermo Cuadra,Match Report,NaN
2,2021-08-28,22:00 (02:00),La Liga,Matchweek 3,Sat,Away,W,1,0,Real Betis,49,22590.0,Karim Benzema,4-3-3,4-2-3-1,Alejandro HernÃ¡ndez,Match Report,NaN
3,2021-09-12,21:00 (01:00),La Liga,Matchweek 4,Sun,Home,W,5,2,Celta Vigo,60,19874.0,Karim Benzema,4-4-2,4-1-3-2,JosÃ© SÃ¡nchez,Match Report,NaN
4,2021-09-15,21:00 (01:00),Champions Lg,Group stage,Wed,Away,W,1,0,it Inter,53,37082.0,Karim Benzema,4-3-3,3-5-2,Daniel Siebert,Match Report,NaN



SOURCE: sportsref_download stats 22-23.xls::table_0
ROWS: 25
COLUMNS: 22


Unnamed: 0_level_0 Unnamed: 1_level_0 Unnamed: 2_level_0 Unnamed: 3_level_0  \
              Player             Nation                Pos                Age   
0      Karim Benzema             fr FRA                 FW               34.0   
1   Thibaut Courtois             be BEL                 GK               30.0   
2   Vinicius JÃºnior             br BRA                 FW               22.0   
3   Antonio RÃ¼diger             de GER                 DF               29.0   
4  Federico Valverde             uy URU              MF,FW               24.0   

  Unnamed: 4_level_0 Playing Time             Performance       ...       \
                  MP       Starts    Min  90s         Gls  Ast  ...   PK   
0                  2            2  210.0  2.3         2.0  0.0  ...  1.0   
1                  2            2  210.0  2.3         0.0  0.0  ...  0.0   
2                  2            2  210.0  2.3         0.0  0.0  ...  0.0   
3                  2            2  210.0  2.3         0.0  0.0  ...  0.0   
4                  2            2  194.0  2.2         0.0  0.0  ...  0.0   

                  Per 90 Minutes                         Unnamed: 21_level_0  
  PKatt CrdY CrdR            Gls  Ast   G+A  G-PK G+A-PK             Matches  
0   1.0  0.0  0.0           0.86  0.0  0.86  0.43   0.43             Matches  
1   0.0  0.0  0.0           0.00  0.0  0.00  0.00   0.00             Matches  
2   0.0  0.0  0.0           0.00  0.0  0.00  0.00   0.00             Matches  
3   0.0  0.0  0.0           0.00  0.0  0.00  0.00   0.00             Matches  
4   0.0  1.0  0.0           0.00  0.0  0.00  0.00   0.00             Matches  

[5 rows x 22 columns]


SOURCE: sportsref_download md 22-23.xls::table_0
ROWS: 59
COLUMNS: 18


,Date,Time,Comp,Round,Day,Venue,Result,GF,GA,Opponent,Poss,Attendance,Captain,Formation,Opp Formation,Referee,Match Report,Notes
0,2022-08-10,22:00 (01:00),Super Cup,UEFA Super Cup,Wed,Home,W,2,0,de Frankfurt,58,31042,Karim Benzema,4-3-3,3-4-3,Michael Oliver,Match Report,NaN
1,2022-08-14,22:00 (02:00),La Liga,Matchweek 1,Sun,Away,W,2,1,AlmerÃ­a,68,14386,Karim Benzema,4-3-3,5-3-2,Juan MartÃ­nez,Match Report,NaN
2,2022-08-20,22:00 (02:00),La Liga,Matchweek 2,Sat,Away,W,4,1,Celta Vigo,49,15681,Karim Benzema,4-3-3,4-1-3-2,JesÃºs Gil,Match Report,NaN
3,2022-08-28,22:00 (02:00),La Liga,Matchweek 3,Sun,Away,W,3,1,Espanyol,64,25778,Karim Benzema,4-3-3,4-1-4-1,Mario Melero,Match Report,NaN
4,2022-09-03,16:15 (20:15),La Liga,Matchweek 4,Sat,Home,W,2,1,Real Betis,50,58579,Karim Benzema,4-3-3,4-2-3-1,JosÃ© SÃ¡nchez,Match Report,NaN


... (34 outputs left out)


SOURCE: sportsref_download md 20-21.xls::table_0
ROWS: 52
COLUMNS: 18


,Date,Time,Comp,Round,Day,Venue,Result,GF,GA,Opponent,Poss,Attendance,Captain,Formation,Opp Formation,Referee,Match Report,Notes
0,2020-09-20,21:00 (01:00),La Liga,Matchweek 2,Sun,Away,D,0,0,Real Sociedad,68,NaN,Sergio Ramos,4-2-3-1,4-4-2,Juan MartÃ­nez,Match Report,NaN
1,2020-09-26,21:00 (01:00),La Liga,Matchweek 3,Sat,Away,W,3,2,Real Betis,54,NaN,Sergio Ramos,4-3-1-2,4-2-3-1,Ricardo de Burgos,Match Report,NaN
2,2020-09-30,21:30 (01:30),La Liga,Matchweek 4,Wed,Home,W,1,0,Valladolid,63,NaN,Sergio Ramos,4-3-1-2,4-2-3-1,CÃ©sar Soto,Match Report,NaN
3,2020-10-04,16:00 (20:00),La Liga,Matchweek 5,Sun,Away,W,2,0,Levante,55,NaN,Sergio Ramos,4-3-3,4-4-2,JosÃ© Luis Munuera,Match Report,NaN
4,2020-10-17,18:30 (22:30),La Liga,Matchweek 6,Sat,Home,L,0,1,CÃ¡diz,75,NaN,Sergio Ramos,4-3-3,4-4-2,Santiago Jaime,Match Report,NaN



SOURCE: sportsref_download stats 21-22.xls::table_0
ROWS: 36
COLUMNS: 22


Unnamed: 0_level_0 Unnamed: 1_level_0 Unnamed: 2_level_0 Unnamed: 3_level_0  \
              Player             Nation                Pos                Age   
0   Thibaut Courtois             be BEL                 GK                 29   
1     Ãder MilitÃ£o             br BRA                 DF                 23   
2   Vinicius JÃºnior             br BRA                 FW                 21   
3        David Alaba             at AUT                 DF                 29   
4           Casemiro             br BRA                 MF                 29   

  Unnamed: 4_level_0 Playing Time               Performance        ...       \
                  MP       Starts     Min   90s         Gls   Ast  ...   PK   
0                 52           52  4770.0  53.0         0.0   0.0  ...  0.0   
1                 50           50  4491.0  49.9         2.0   2.0  ...  0.0   
2                 52           47  4253.0  47.3        22.0  16.0  ...  0.0   
3                 46           46  4066.0  45.2         3.0   4.0  ...  0.0   
4                 48           46  3918.0  43.5         1.0   3.0  ...  0.0   

                   Per 90 Minutes                          Unnamed: 21_level_0  
  PKatt  CrdY CrdR            Gls   Ast   G+A  G-PK G+A-PK             Matches  
0   0.0   1.0  0.0           0.00  0.00  0.00  0.00   0.00             Matches  
1   0.0   9.0  1.0           0.04  0.04  0.08  0.04   0.08             Matches  
2   0.0   7.0  0.0           0.47  0.34  0.80  0.47   0.80             Matches  
3   0.0   4.0  0.0           0.07  0.09  0.15  0.07   0.15             Matches  
4   0.0  16.0  0.0           0.02  0.07  0.09  0.02   0.09             Matches  

[5 rows x 22 columns]

In [32]:

# ============================================================
# PHASE 4.3 — FBREF DATA VALIDATION & SCHEMA NORMALISATION
# Real Madrid | 2015-16 to 2025-26
# ============================================================

import pandas as pd
import re
from pathlib import Path

# 1. Confirm extracted tables are available
if "fbref_tables" not in globals() or not fbref_tables:
    raise RuntimeError(
        "fbref_tables not found. Run the FBref extraction cell first."
    )

# 2. Create a separate audit folder
BASE_DIR = Path("/content/real_madrid_project")
AUDIT_DIR = BASE_DIR / "Data/Processed/03_fbref_validation"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

# 3. Detect common character-encoding errors
SUSPICIOUS_MARKERS = ["Ã", "Â", "â€", "ðŸ", "�"]

def has_encoding_issue(value):
    return isinstance(value, str) and any(
        marker in value for marker in SUSPICIOUS_MARKERS
    )

def repair_encoding(value):
    if not isinstance(value, str):
        return value

    if not has_encoding_issue(value):
        return value

    try:
        candidate = value.encode("latin1").decode("utf-8")

        if sum(candidate.count(m) for m in SUSPICIOUS_MARKERS) < \
           sum(value.count(m) for m in SUSPICIOUS_MARKERS):
            return candidate

    except (UnicodeEncodeError, UnicodeDecodeError):
        pass

    return value

# 4. Flatten multi-level FBref column headers
def flatten_column(column):

    if isinstance(column, tuple):
        parts = [
            str(part).strip()
            for part in column
            if str(part).strip()
            and not str(part).startswith("Unnamed:")
            and str(part).lower() not in ["nan", "none"]
        ]

        return "_".join(parts) if parts else "unnamed"

    return str(column).strip()

def make_unique(columns):

    counts = {}
    result = []

    for column in columns:
        counts[column] = counts.get(column, 0) + 1

        if counts[column] > 1:
            result.append(f"{column}__{counts[column]}")
        else:
            result.append(column)

    return result

# 5. Prepare separate normalised copies
fbref_normalized = {}

validation_reports = []
schema_reports = []
encoding_reports = []

for key, original_df in fbref_tables.items():

    source_file = key.split("::")[0]

    season_match = re.search(r"(\d{2}-\d{2})", source_file)
    season = season_match.group(1) if season_match else "UNKNOWN"

    dataset_type = (
        "player_season_stats"
        if "stats" in source_file.lower()
        else "match_log"
    )

    df = original_df.copy()

    # Count encoding issues before repair
    before_issues = sum(
        has_encoding_issue(value)
        for value in df.to_numpy().flatten()
    )

    # Flatten column headers
    flattened = [flatten_column(c) for c in df.columns]
    df.columns = make_unique(flattened)

    # Repair text values on the copy
    replacements = []

    for column in df.columns:

        repaired_values = []

        for value in df[column]:

            repaired = repair_encoding(value)

            if isinstance(value, str) and repaired != value:
                if len(replacements) < 5:
                    replacements.append({
                        "column": column,
                        "original": value,
                        "corrected": repaired
                    })

            repaired_values.append(repaired)

        df[column] = repaired_values

    after_issues = sum(
        has_encoding_issue(value)
        for value in df.to_numpy().flatten()
    )

    # Save the normalised copy in memory
    fbref_normalized[key] = df

    # Determine required fields
    if dataset_type == "match_log":
        required = [
            "Date", "Comp", "Opponent", "Result", "GF", "GA"
        ]
    else:
        required = ["Player", "MP", "Min"]

    missing_columns = [
        col for col in required if col not in df.columns
    ]

    missing_player_names = (
        df["Player"].isna().sum()
        if "Player" in df.columns
        else None
    )

    validation_reports.append({
        "source": source_file,
        "season": season,
        "dataset_type": dataset_type,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_cells": int(df.isna().sum().sum()),
        "exact_duplicate_rows": int(df.duplicated().sum()),
        "missing_required_columns": ", ".join(missing_columns),
        "missing_player_names": missing_player_names,
        "encoding_issues_before": before_issues,
        "encoding_issues_after": after_issues,
        "encoding_repairs": before_issues - after_issues
    })

    # Record every table's schema
    for position, column in enumerate(df.columns):

        schema_reports.append({
            "source": source_file,
            "season": season,
            "dataset_type": dataset_type,
            "column_position": position,
            "column_name": column,
            "data_type": str(df[column].dtype)
        })

    # Record sample encoding corrections
    for replacement in replacements:

        encoding_reports.append({
            "source": source_file,
            "season": season,
            **replacement
        })

# 6. Create audit reports
validation_df = pd.DataFrame(validation_reports)
schema_df = pd.DataFrame(schema_reports)
encoding_df = pd.DataFrame(encoding_reports)

# 7. Save audit reports only — master datasets untouched
validation_df.to_csv(
    AUDIT_DIR / "fbref_validation_report.csv",
    index=False,
    encoding="utf-8-sig"
)

schema_df.to_csv(
    AUDIT_DIR / "fbref_schema_report.csv",
    index=False,
    encoding="utf-8-sig"
)

encoding_df.to_csv(
    AUDIT_DIR / "fbref_encoding_report.csv",
    index=False,
    encoding="utf-8-sig"
)

# 8. Display results
print("=" * 65)
print("REAL MADRID — FBREF VALIDATION REPORT")
print("=" * 65)

print("\nTOTAL TABLES:", len(fbref_normalized))
print("PLAYER STAT TABLES:",
      (validation_df.dataset_type == "player_season_stats").sum())
print("MATCH LOG TABLES:",
      (validation_df.dataset_type == "match_log").sum())

print("\nSEASON-BY-SEASON VALIDATION")
display(validation_df)

print("\nCOLUMN SCHEMA")
display(schema_df)

print("\nENCODING CORRECTIONS")
display(encoding_df)

print("\nAUDIT FILES SAVED TO:")
print(AUDIT_DIR)

print("\nMASTER DATASETS HAVE NOT BEEN MODIFIED.")


REAL MADRID — FBREF VALIDATION REPORT

TOTAL TABLES: 22
PLAYER STAT TABLES: 11
MATCH LOG TABLES: 11

SEASON-BY-SEASON VALIDATION


,source,season,dataset_type,rows,columns,missing_cells,exact_duplicate_rows,missing_required_columns,missing_player_names,encoding_issues_before,encoding_issues_after,encoding_repairs
0,sportsref_download md 21-22.xls,21-22,match_log,56,18,52,0,,NaN,44,0,44
1,sportsref_download stats 22-23.xls,22-23,player_season_stats,25,22,111,0,Min,0.0,8,0,8
2,sportsref_download md 22-23.xls,22-23,match_log,59,18,52,0,,NaN,43,0,43
3,sportsref_download stats 23-24.xls,23-24,player_season_stats,38,22,135,0,Min,0.0,15,0,15
4,sportsref_download md 23-24.xls,23-24,match_log,55,18,47,0,,NaN,45,0,45
5,sportsref_download stats 24-25.xls,24-25,player_season_stats,38,22,105,0,Min,0.0,16,0,16
6,sportsref_download md 24-25.xls,24-25,match_log,61,18,51,0,,NaN,70,0,70
7,sportsref_download stats 25-26.xls,25-26,player_season_stats,44,22,120,0,Min,0.0,16,0,16
8,sportsref_download md 25-26.xls,25-26,match_log,56,18,50,0,,NaN,45,0,45
9,sportsref_download stats 15-16.xls,15-16,player_season_stats,29,22,45,0,"MP, Min",0.0,9,0,9



COLUMN SCHEMA


,source,season,dataset_type,column_position,column_name,data_type
0,sportsref_download md 21-22.xls,21-22,match_log,0,Date,object
1,sportsref_download md 21-22.xls,21-22,match_log,1,Time,object
2,sportsref_download md 21-22.xls,21-22,match_log,2,Comp,object
3,sportsref_download md 21-22.xls,21-22,match_log,3,Round,object
4,sportsref_download md 21-22.xls,21-22,match_log,4,Day,object
...,...,...,...,...,...,...
435,sportsref_download stats 21-22.xls,21-22,player_season_stats,17,Per 90 Minutes_Ast,float64
436,sportsref_download stats 21-22.xls,21-22,player_season_stats,18,Per 90 Minutes_G+A,float64
437,sportsref_download stats 21-22.xls,21-22,player_season_stats,19,Per 90 Minutes_G-PK,float64
438,sportsref_download stats 21-22.xls,21-22,player_season_stats,20,Per 90 Minutes_G+A-PK,float64



ENCODING CORRECTIONS


,source,season,column,original,corrected
0,sportsref_download md 21-22.xls,21-22,Comp,Supercopa de EspaÃ±a,Supercopa de España
1,sportsref_download md 21-22.xls,21-22,Comp,Supercopa de EspaÃ±a,Supercopa de España
2,sportsref_download md 21-22.xls,21-22,Opponent,AlavÃ©s,Alavés
3,sportsref_download md 21-22.xls,21-22,Opponent,AtlÃ©tico Madrid,Atlético Madrid
4,sportsref_download md 21-22.xls,21-22,Opponent,CÃ¡diz,Cádiz
...,...,...,...,...,...
105,sportsref_download stats 21-22.xls,21-22,Player,Ãder MilitÃ£o,Éder Militão
106,sportsref_download stats 21-22.xls,21-22,Player,Vinicius JÃºnior,Vinicius Júnior
107,sportsref_download stats 21-22.xls,21-22,Player,Lucas VÃ¡zquez,Lucas Vázquez
108,sportsref_download stats 21-22.xls,21-22,Player,JesÃºs Vallejo,Jesús Vallejo



AUDIT FILES SAVED TO:
/content/real_madrid_project/Data/Processed/03_fbref_validation

MASTER DATASETS HAVE NOT BEEN MODIFIED.


In [33]:

# ============================================================
# PHASE 4.3 — CELL 19
# DETAILED SCHEMA AND MISSINGNESS INVESTIGATION
# ============================================================

import pandas as pd
import re
from pathlib import Path

if "fbref_normalized" not in globals():
    raise RuntimeError(
        "Run the previous FBref validation cell first."
    )

AUDIT_DIR = Path(
    "/content/real_madrid_project/Data/Processed/03_fbref_validation"
)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

column_audit = []
schema_comparison = []

stats_total = 0
match_total = 0

print("=" * 70)
print("FBREF DETAILED SCHEMA INVESTIGATION")
print("=" * 70)

for key, df in fbref_normalized.items():

    filename = key.split("::")[0]

    season_match = re.search(
        r"(\d{2}-\d{2})", filename
    )

    season = (
        season_match.group(1)
        if season_match else "UNKNOWN"
    )

    is_stats = "stats" in filename.lower()

    dataset_type = (
        "player_season_stats"
        if is_stats else "match_log"
    )

    if is_stats:
        stats_total += len(df)
    else:
        match_total += len(df)

    print("\n" + "=" * 70)
    print("SOURCE:", filename)
    print("SEASON:", season)
    print("TYPE:", dataset_type)
    print("ROWS:", len(df))
    print("COLUMNS:", len(df.columns))

    print("\nACTUAL COLUMN NAMES:")

    for i, col in enumerate(df.columns):
        print(f"{i:02d}. {col}")

    # Check player statistics column aliases
    if is_stats:

        print("\nPLAYER COLUMN CANDIDATES:")
        print([
            c for c in df.columns
            if c.lower() == "player"
            or c.lower().endswith("_player")
        ])

        print("MP COLUMN CANDIDATES:")
        print([
            c for c in df.columns
            if c.lower() == "mp"
            or c.lower().endswith("_mp")
        ])

        print("MIN COLUMN CANDIDATES:")
        print([
            c for c in df.columns
            if c.lower() == "min"
            or c.lower().endswith("_min")
        ])

    # Missingness by column
    for col in df.columns:

        missing = int(df[col].isna().sum())

        column_audit.append({
            "source": filename,
            "season": season,
            "dataset_type": dataset_type,
            "column": col,
            "rows": len(df),
            "missing_values": missing,
            "missing_percentage": round(
                missing / len(df) * 100, 2
            ) if len(df) else 0,
            "dtype": str(df[col].dtype)
        })

        schema_comparison.append({
            "season": season,
            "dataset_type": dataset_type,
            "column": col
        })

# Create audit tables
column_audit_df = pd.DataFrame(column_audit)
schema_df = pd.DataFrame(schema_comparison)

print("\n" + "=" * 70)
print("TOTAL SOURCE ROWS")
print("=" * 70)

print("Player statistics:", stats_total)
print("Match logs:", match_total)

print("\n" + "=" * 70)
print("MISSINGNESS SUMMARY")
print("=" * 70)

missing_summary = (
    column_audit_df[
        column_audit_df["missing_values"] > 0
    ]
    .sort_values(
        ["dataset_type", "missing_percentage"],
        ascending=[True, False]
    )
)

display(missing_summary)

# Compare column availability across seasons
print("\n" + "=" * 70)
print("COLUMN AVAILABILITY BY DATASET TYPE")
print("=" * 70)

column_coverage = (
    schema_df.groupby(
        ["dataset_type", "column"]
    )["season"]
    .nunique()
    .reset_index(name="seasons_present")
)

display(
    column_coverage.sort_values(
        ["dataset_type", "column"]
    )
)

# Save reports
column_audit_df.to_csv(
    AUDIT_DIR / "fbref_column_missingness_detailed.csv",
    index=False,
    encoding="utf-8-sig"
)

column_coverage.to_csv(
    AUDIT_DIR / "fbref_column_season_coverage.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nREPORTS SAVED TO:")
print(AUDIT_DIR)

print("\nNO MASTER DATASETS MODIFIED.")


FBREF DETAILED SCHEMA INVESTIGATION

SOURCE: sportsref_download md 21-22.xls
SEASON: 21-22
TYPE: match_log
ROWS: 56
COLUMNS: 18

ACTUAL COLUMN NAMES:
00. Date
01. Time
02. Comp
03. Round
04. Day
05. Venue
06. Result
07. GF
08. GA
09. Opponent
10. Poss
11. Attendance
12. Captain
13. Formation
14. Opp Formation
15. Referee
16. Match Report
17. Notes

SOURCE: sportsref_download stats 22-23.xls
SEASON: 22-23
TYPE: player_season_stats
ROWS: 25
COLUMNS: 22

ACTUAL COLUMN NAMES:
00. Player
01. Nation
02. Pos
03. Age
04. MP
05. Playing Time_Starts
06. Playing Time_Min
07. Playing Time_90s
08. Performance_Gls
09. Performance_Ast
10. Performance_G+A
11. Performance_G-PK
12. Performance_PK
13. Performance_PKatt
14. Performance_CrdY
15. Performance_CrdR
16. Per 90 Minutes_Gls
17. Per 90 Minutes_Ast
18. Per 90 Minutes_G+A
19. Per 90 Minutes_G-PK
20. Per 90 Minutes_G+A-PK
21. Matches

PLAYER COLUMN CANDIDATES:
['Player']
MP COLUMN CANDIDATES:
['MP']
MIN COLUMN CANDIDATES:
['Playing Time_Min']

SOURC

,source,season,dataset_type,column,rows,missing_values,missing_percentage,dtype
411,sportsref_download md 20-21.xls,20-21,match_log,Attendance,52,52,100.00,float64
377,sportsref_download md 19-20.xls,19-20,match_log,Notes,51,48,94.12,object
177,sportsref_download md 25-26.xls,25-26,match_log,Notes,56,50,89.29,object
417,sportsref_download md 20-21.xls,20-21,match_log,Notes,52,45,86.54,object
17,sportsref_download md 21-22.xls,21-22,match_log,Notes,56,48,85.71,object
...,...,...,...,...,...,...,...,...
234,sportsref_download stats 16-17.xls,16-17,player_season_stats,Per 90 Minutes_Gls,28,1,3.57,float64
235,sportsref_download stats 16-17.xls,16-17,player_season_stats,Per 90 Minutes_Ast,28,1,3.57,float64
236,sportsref_download stats 16-17.xls,16-17,player_season_stats,Per 90 Minutes_G+A,28,1,3.57,float64
237,sportsref_download stats 16-17.xls,16-17,player_season_stats,Per 90 Minutes_G-PK,28,1,3.57,float64



COLUMN AVAILABILITY BY DATASET TYPE


,dataset_type,column,seasons_present
0,match_log,Attendance,11
1,match_log,Captain,11
2,match_log,Comp,11
3,match_log,Date,11
4,match_log,Day,11
5,match_log,Formation,11
6,match_log,GA,11
7,match_log,GF,11
8,match_log,Match Report,11
9,match_log,Notes,11



REPORTS SAVED TO:
/content/real_madrid_project/Data/Processed/03_fbref_validation

NO MASTER DATASETS MODIFIED.


In [34]:

# ============================================================
# PHASE 4.4 — FBREF CONSOLIDATION & MASTER RECONCILIATION
# ============================================================

import pandas as pd
import re
from pathlib import Path
from IPython.display import display

BASE = Path("/content/real_madrid_project")
CLEANED = BASE / "Data" / "Cleaned"
AUDIT = BASE / "Data" / "Processed" / "03_fbref_validation"

AUDIT.mkdir(parents=True, exist_ok=True)

if "fbref_normalized" not in globals():
    raise RuntimeError("Run the previous FBref extraction cells first.")

player_tables = []
match_tables = []

# ------------------------------------------------------------
# 1. STANDARDIZE PLAYER STATISTICS
# ------------------------------------------------------------

player_rename = {
    "Playing Time_MP": "MP",
    "Playing Time_Starts": "Starts",
    "Playing Time_Min": "Minutes",
    "Playing Time_90s": "Nineties",
    "Performance_Gls": "Goals",
    "Performance_Ast": "Assists",
    "Performance_G+A": "Goals_Assists",
    "Performance_G-PK": "Non_Penalty_Goals",
    "Performance_PK": "Penalty_Goals",
    "Performance_PKatt": "Penalty_Attempts",
    "Performance_CrdY": "Yellow_Cards",
    "Performance_CrdR": "Red_Cards",
    "Per 90 Minutes_Gls": "Goals_Per_90",
    "Per 90 Minutes_Ast": "Assists_Per_90",
    "Per 90 Minutes_G+A": "Goals_Assists_Per_90",
    "Per 90 Minutes_G-PK": "Non_Penalty_Goals_Per_90",
    "Per 90 Minutes_G+A-PK": "Non_Penalty_Goals_Assists_Per_90"
}

match_rename = {
    "Date": "match_date",
    "Time": "match_time",
    "Comp": "competition",
    "Round": "round",
    "Day": "day",
    "Venue": "venue",
    "Result": "result",
    "GF": "goals_for",
    "GA": "goals_against",
    "Opponent": "opponent",
    "Poss": "possession",
    "Attendance": "attendance",
    "Captain": "captain",
    "Formation": "formation",
    "Opp Formation": "opponent_formation",
    "Referee": "referee",
    "Match Report": "match_report",
    "Notes": "notes"
}

for key, original_df in fbref_normalized.items():

    filename = key.split("::")[0]

    season_match = re.search(r"(\d{2}-\d{2})", filename)

    if not season_match:
        print("WARNING: Could not identify season:", filename)
        continue

    season = season_match.group(1)

    df = original_df.copy()

    if "stats" in filename.lower():

        # Normalize MP and other inconsistent names
        df = df.rename(columns=player_rename)

        df["season"] = season
        df["source_file"] = filename

        player_tables.append(df)

    elif "md" in filename.lower():

        df = df.rename(columns=match_rename)

        df["season"] = season
        df["source_file"] = filename

        match_tables.append(df)

# ------------------------------------------------------------
# 2. COMBINE ALL SEASONS
# ------------------------------------------------------------

players = pd.concat(
    player_tables,
    ignore_index=True,
    sort=False
)

matches = pd.concat(
    match_tables,
    ignore_index=True,
    sort=False
)

print("=" * 65)
print("CONSOLIDATION RESULTS")
print("=" * 65)

print("Player records:", len(players))
print("Player columns:", len(players.columns))

print("Match records:", len(matches))
print("Match columns:", len(matches.columns))

# ------------------------------------------------------------
# 3. CHECK EXPECTED SEASON COVERAGE
# ------------------------------------------------------------

expected_seasons = [
    f"{str(y)[-2:]}-{str(y+1)[-2:]}"
    for y in range(2015, 2026)
]

season_summary = pd.DataFrame({
    "season": expected_seasons
})

player_counts = (
    players.groupby("season")
    .size()
    .rename("fbref_player_rows")
)

match_counts = (
    matches.groupby("season")
    .size()
    .rename("fbref_match_rows")
)

season_summary = season_summary.merge(
    player_counts,
    on="season",
    how="left"
).merge(
    match_counts,
    on="season",
    how="left"
)

season_summary = season_summary.fillna(0)

print("\nSEASON-BY-SEASON FBREF COVERAGE")
display(season_summary)

# ------------------------------------------------------------
# 4. CHECK PLAYER DUPLICATES
# ------------------------------------------------------------

player_duplicates = players[
    players.duplicated(
        subset=["season", "Player"],
        keep=False
    )
].sort_values(["season", "Player"])

print("\nDUPLICATE PLAYER-SEASON RECORDS")
print("Rows involved:", len(player_duplicates))

if len(player_duplicates):
    display(player_duplicates[
        ["season", "Player", "MP", "Minutes", "source_file"]
    ])

# ------------------------------------------------------------
# 5. CHECK MATCH DUPLICATES
# ------------------------------------------------------------

match_duplicates = matches[
    matches.duplicated(
        subset=[
            "season",
            "match_date",
            "competition",
            "opponent"
        ],
        keep=False
    )
].sort_values(
    ["season", "match_date", "competition", "opponent"]
)

print("\nPOTENTIAL MATCH DUPLICATES")
print("Rows involved:", len(match_duplicates))

if len(match_duplicates):
    display(match_duplicates[
        [
            "season",
            "match_date",
            "competition",
            "opponent",
            "result"
        ]
    ])

# ------------------------------------------------------------
# 6. INSPECT EXISTING MASTER DATASETS
# ------------------------------------------------------------

master_paths = {
    "player_season_stats":
        CLEANED / "Real_Madrid_Player_Season_Stats.csv",

    "matches":
        CLEANED / "Real_Madrid_All_Matches.csv"
}

master_data = {}

for name, path in master_paths.items():

    print("\nMASTER:", name)

    if path.exists():

        master_df = pd.read_csv(path)

        master_data[name] = master_df

        print("Rows:", len(master_df))
        print("Columns:", list(master_df.columns))

    else:
        print("File not found:", path)

# ------------------------------------------------------------
# 7. COMPARE MASTER SEASON COUNTS
# ------------------------------------------------------------

def get_master_season_counts(df):

    season_col = next(
        (
            c for c in df.columns
            if c.lower().strip() in ["season", "season_id"]
        ),
        None
    )

    if season_col:
        return (
            df.groupby(season_col)
            .size()
            .reset_index(name="master_rows")
            .rename(columns={season_col: "season"})
        )

    # Derive season from dates if no season field exists
    date_col = next(
        (
            c for c in df.columns
            if c.lower().strip() in [
                "match_date", "date", "transfer_date"
            ]
        ),
        None
    )

    if date_col:

        dates = pd.to_datetime(
            df[date_col],
            errors="coerce",
            dayfirst=True
        )

        start_year = dates.dt.year.where(
            dates.dt.month >= 7,
            dates.dt.year - 1
        )

        derived = (
            start_year.astype("Int64").astype(str).str[-2:]
            + "-"
            + (start_year + 1).astype("Int64").astype(str).str[-2:]
        )

        temp = pd.DataFrame({"season": derived})

        return (
            temp.groupby("season")
            .size()
            .reset_index(name="master_rows")
        )

    return None


for name, fbref_df in [
    ("player_season_stats", players),
    ("matches", matches)
]:

    if name not in master_data:
        continue

    print("\n" + "=" * 65)
    print("MASTER RECONCILIATION:", name)
    print("=" * 65)

    master_counts = get_master_season_counts(
        master_data[name]
    )

    if master_counts is None:

        print("No usable season/date field found.")
        continue

    fbref_counts = (
        fbref_df.groupby("season")
        .size()
        .reset_index(name="fbref_rows")
    )

    comparison = master_counts.merge(
        fbref_counts,
        on="season",
        how="outer"
    ).fillna(0)

    comparison["difference_master_minus_fbref"] = (
        comparison["master_rows"] - comparison["fbref_rows"]
    )

    display(comparison.sort_values("season"))

    comparison.to_csv(
        AUDIT / f"{name}_season_reconciliation.csv",
        index=False
    )

# ------------------------------------------------------------
# 8. SAVE SEPARATE STAGING DATASETS
# ------------------------------------------------------------

players.to_csv(
    AUDIT / "FBref_Player_Season_Stats_Staging.csv",
    index=False,
    encoding="utf-8-sig"
)

matches.to_csv(
    AUDIT / "FBref_Match_Logs_Staging.csv",
    index=False,
    encoding="utf-8-sig"
)

season_summary.to_csv(
    AUDIT / "FBref_Season_Coverage.csv",
    index=False
)

print("\n" + "=" * 65)
print("PHASE 4.4 COMPLETED")
print("=" * 65)

print("Staging files saved to:")
print(AUDIT)

print("\nEXISTING MASTER DATASETS WERE NOT MODIFIED.")


CONSOLIDATION RESULTS
Player records: 375
Player columns: 24
Match records: 617
Match columns: 20

SEASON-BY-SEASON FBREF COVERAGE


,season,fbref_player_rows,fbref_match_rows
0,15-16,29,54
1,16-17,28,58
2,17-18,32,60
3,18-19,36,55
4,19-20,32,51
5,20-21,37,52
6,21-22,36,56
7,22-23,25,59
8,23-24,38,55
9,24-25,38,61



DUPLICATE PLAYER-SEASON RECORDS
Rows involved: 0

POTENTIAL MATCH DUPLICATES
Rows involved: 0

MASTER: player_season_stats
File not found: /content/real_madrid_project/Data/Cleaned/Real_Madrid_Player_Season_Stats.csv

MASTER: matches
File not found: /content/real_madrid_project/Data/Cleaned/Real_Madrid_All_Matches.csv

PHASE 4.4 COMPLETED
Staging files saved to:
/content/real_madrid_project/Data/Processed/03_fbref_validation

EXISTING MASTER DATASETS WERE NOT MODIFIED.


In [35]:

# ============================================================
# PHASE 4.4 — CELL 21
# LOCATE EXISTING MASTER DATASETS
# ============================================================

from pathlib import Path

SEARCH_ROOT = Path("/content")

target_files = [
    "Real_Madrid_Player_Season_Stats.csv",
    "Real_Madrid_All_Matches.csv",
    "Real_Madrid_Player_Competition_Stats.csv",
    "Real_Madrid_Match_Team_Stats.csv",
    "Real_Madrid_Transfers.csv",
    "Real_Madrid_Financials.csv"
]

print("=" * 65)
print("SEARCHING FOR EXISTING MASTER DATASETS")
print("=" * 65)

found_files = {}

for filename in target_files:

    matches = list(SEARCH_ROOT.rglob(filename))

    found_files[filename] = matches

    print("\nFILE:", filename)

    if matches:

        for path in matches:
            print("FOUND:", path)
            print("SIZE:", path.stat().st_size, "bytes")

    else:
        print("NOT FOUND")

# Also locate the master ZIP if present
print("\n" + "=" * 65)
print("SEARCHING FOR MASTER ZIP ARCHIVES")
print("=" * 65)

zip_matches = list(
    SEARCH_ROOT.rglob(
        "Real_Madrid_Master_Datasets*.zip"
    )
)

for path in zip_matches:
    print("FOUND ZIP:", path)

# Save location report
location_report = []

for filename, paths in found_files.items():

    if paths:

        for path in paths:
            location_report.append({
                "filename": filename,
                "location": str(path),
                "status": "FOUND"
            })

    else:

        location_report.append({
            "filename": filename,
            "location": "",
            "status": "NOT FOUND"
        })

import pandas as pd

location_df = pd.DataFrame(location_report)

display(location_df)

location_df.to_csv(
    "/content/master_dataset_location_report.csv",
    index=False
)

print("\nLOCATION AUDIT COMPLETED.")


SEARCHING FOR EXISTING MASTER DATASETS

FILE: Real_Madrid_Player_Season_Stats.csv
FOUND: /content/Real_Madrid_Player_Season_Stats.csv
SIZE: 92905 bytes
FOUND: /content/real_madrid_project/Data/Cleaned/Real_Madrid_Master_Datasets_2026-11-18_UPDATED/Real_Madrid_Player_Season_Stats.csv
SIZE: 92905 bytes

FILE: Real_Madrid_All_Matches.csv
FOUND: /content/Real_Madrid_All_Matches.csv
SIZE: 242132 bytes
FOUND: /content/real_madrid_project/Data/Cleaned/Real_Madrid_Master_Datasets_2026-11-18_UPDATED/Real_Madrid_All_Matches.csv
SIZE: 242132 bytes

FILE: Real_Madrid_Player_Competition_Stats.csv
FOUND: /content/Real_Madrid_Player_Competition_Stats.csv
SIZE: 329737 bytes
FOUND: /content/real_madrid_project/Data/Cleaned/Real_Madrid_Master_Datasets_2026-11-18_UPDATED/Real_Madrid_Player_Competition_Stats.csv
SIZE: 329737 bytes

FILE: Real_Madrid_Match_Team_Stats.csv
FOUND: /content/Real_Madrid_Match_Team_Stats.csv
SIZE: 237636 bytes
FOUND: /content/real_madrid_project/Data/Cleaned/Real_Madrid_Master_D

,filename,location,status
0,Real_Madrid_Player_Season_Stats.csv,/content/Real_Madrid_Player_Season_Stats.csv,FOUND
1,Real_Madrid_Player_Season_Stats.csv,/content/real_madrid_project/Data/Cleaned/Real...,FOUND
2,Real_Madrid_All_Matches.csv,/content/Real_Madrid_All_Matches.csv,FOUND
3,Real_Madrid_All_Matches.csv,/content/real_madrid_project/Data/Cleaned/Real...,FOUND
4,Real_Madrid_Player_Competition_Stats.csv,/content/Real_Madrid_Player_Competition_Stats.csv,FOUND
5,Real_Madrid_Player_Competition_Stats.csv,/content/real_madrid_project/Data/Cleaned/Real...,FOUND
6,Real_Madrid_Match_Team_Stats.csv,/content/Real_Madrid_Match_Team_Stats.csv,FOUND
7,Real_Madrid_Match_Team_Stats.csv,/content/real_madrid_project/Data/Cleaned/Real...,FOUND
8,Real_Madrid_Transfers.csv,/content/Real_Madrid_Transfers.csv,FOUND
9,Real_Madrid_Transfers.csv,/content/real_madrid_project/Data/Cleaned/Real...,FOUND



LOCATION AUDIT COMPLETED.


In [36]:

# ============================================================
# PHASE 4.4 — CELL 22
# MASTER DATASET RECONCILIATION
# READ-ONLY AUDIT
# ============================================================

import pandas as pd
import hashlib
import re
from pathlib import Path

BASE = Path("/content/real_madrid_project")

MASTER_DIR = (
    BASE / "Data/Cleaned/"
    "Real_Madrid_Master_Datasets_2026-11-18_UPDATED"
)

AUDIT_DIR = (
    BASE / "Data/Processed/03_fbref_validation"
)

# ------------------------------------------------------------
# 1. LOAD EXISTING MASTER DATASETS
# ------------------------------------------------------------

master_files = {
    "player_season_stats":
        "Real_Madrid_Player_Season_Stats.csv",

    "matches":
        "Real_Madrid_All_Matches.csv",

    "player_competition_stats":
        "Real_Madrid_Player_Competition_Stats.csv",

    "match_team_stats":
        "Real_Madrid_Match_Team_Stats.csv",

    "transfers":
        "Real_Madrid_Transfers.csv",

    "financials":
        "Real_Madrid_Financials.csv"
}

master_data = {}

for name, filename in master_files.items():

    path = MASTER_DIR / filename

    if not path.exists():
        raise FileNotFoundError(
            f"Missing master dataset: {path}"
        )

    df = pd.read_csv(path, encoding="utf-8-sig")

    master_data[name] = df

    print("\n" + "=" * 65)
    print("DATASET:", name)
    print("ROWS:", len(df))
    print("COLUMNS:", len(df.columns))
    print("FIELDS:", df.columns.tolist())

# ------------------------------------------------------------
# 2. VERIFY ROOT AND NESTED COPIES
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()


print("\n" + "=" * 65)
print("DUPLICATE FILE VERIFICATION")
print("=" * 65)

for name in [
    "Real_Madrid_Player_Season_Stats.csv",
    "Real_Madrid_All_Matches.csv"
]:

    root_copy = Path("/content") / name
    nested_copy = MASTER_DIR / name

    if root_copy.exists():

        same_hash = (
            sha256_file(root_copy)
            == sha256_file(nested_copy)
        )

        print(name, "IDENTICAL:", same_hash)

# ------------------------------------------------------------
# 3. NORMALISE SEASON LABELS FOR COMPARISON
# ------------------------------------------------------------

def normalize_season(value):

    if pd.isna(value):
        return None

    value = str(value).strip()

    match = re.search(
        r"(?<!\d)(\d{2}|\d{4})[-/](\d{2}|\d{4})(?!\d)",
        value
    )

    if not match:
        return value

    start = match.group(1)[-2:]
    end = match.group(2)[-2:]

    return f"{start}-{end}"


def find_season_column(df):

    for col in df.columns:

        if col.lower().strip() in [
            "season",
            "season_id",
            "season_label"
        ]:
            return col

    return None


# ------------------------------------------------------------
# 4. COMPARE SEASON-BY-SEASON RECORD COUNTS
# ------------------------------------------------------------

def season_counts(df, source_name):

    col = find_season_column(df)

    if col is None:

        print(
            f"WARNING: No season column found in {source_name}"
        )

        return None

    seasons = df[col].apply(normalize_season)

    return (
        seasons.value_counts(dropna=False)
        .rename_axis("season")
        .reset_index(name=f"{source_name}_rows")
    )


comparisons = {}

for dataset_name, staging_filename in [
    (
        "player_season_stats",
        "FBref_Player_Season_Stats_Staging.csv"
    ),
    (
        "matches",
        "FBref_Match_Logs_Staging.csv"
    )
]:

    staging_path = AUDIT_DIR / staging_filename

    if not staging_path.exists():
        raise FileNotFoundError(staging_path)

    staging_df = pd.read_csv(
        staging_path,
        encoding="utf-8-sig"
    )

    master_df = master_data[dataset_name]

    master_counts = season_counts(
        master_df,
        "master"
    )

    fbref_counts = season_counts(
        staging_df,
        "fbref"
    )

    print("\n" + "=" * 65)
    print("RECONCILIATION:", dataset_name)
    print("=" * 65)

    if master_counts is None or fbref_counts is None:
        continue

    comparison = master_counts.merge(
        fbref_counts,
        on="season",
        how="outer"
    ).fillna(0)

    comparison["difference_master_minus_fbref"] = (
        comparison["master_rows"]
        - comparison["fbref_rows"]
    )

    comparison = comparison.sort_values("season")

    display(comparison)

    comparisons[dataset_name] = comparison

    comparison.to_csv(
        AUDIT_DIR /
        f"{dataset_name}_master_reconciliation.csv",
        index=False
    )

# ------------------------------------------------------------
# 5. FINAL ROW COUNT COMPARISON
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("OVERALL RECONCILIATION SUMMARY")
print("=" * 65)

for name, staging_name in [
    (
        "player_season_stats",
        "FBref_Player_Season_Stats_Staging.csv"
    ),
    (
        "matches",
        "FBref_Match_Logs_Staging.csv"
    )
]:

    staging = pd.read_csv(
        AUDIT_DIR / staging_name
    )

    master = master_data[name]

    print(f"\n{name}")
    print("Master records:", len(master))
    print("FBref records:", len(staging))
    print("Difference:", len(master) - len(staging))

print("\nMASTER RECONCILIATION AUDIT COMPLETED.")
print("No original master datasets were modified.")



DATASET: player_season_stats
ROWS: 386
COLUMNS: 29
FIELDS: ['player_id', 'player_key_candidate', 'player_name', 'nation', 'position', 'age', 'season', 'season_role', 'competition_scope', 'appearances', 'starts', 'minutes', 'nineties', 'goals', 'assists', 'goal_contributions', 'non_penalty_goals', 'penalties_scored', 'penalties_attempted', 'yellow_cards', 'red_cards', 'goals_per90', 'assists_per90', 'goal_contributions_per90', 'non_penalty_goals_per90', 'goal_contributions_minus_penalties_per90', 'source_name', 'source_file', 'source_row']

DATASET: matches
ROWS: 630
COLUMNS: 47
FIELDS: ['match_id', 'season', 'season_role', 'match_date', 'competition', 'competition_edition', 'round', 'matchday', 'home_team', 'away_team', 'opponent', 'real_madrid_venue', 'real_madrid_goals_for', 'real_madrid_goals_against', 'home_goals', 'away_goals', 'home_goals_90', 'away_goals_90', 'home_goals_after_extra_time', 'away_goals_after_extra_time', 'home_penalties', 'away_penalties', 'result_code_source', 

,season,master_rows,fbref_rows,difference_master_minus_fbref
0,15-16,29,29,0
1,16-17,28,28,0
2,17-18,32,32,0
3,18-19,36,36,0
4,19-20,32,32,0
5,20-21,37,37,0
6,21-22,36,36,0
7,22-23,36,25,11
8,23-24,38,38,0
9,24-25,38,38,0



RECONCILIATION: matches


,season,master_rows,fbref_rows,difference_master_minus_fbref
0,15-16,53,54,-1
1,16-17,60,58,2
2,17-18,62,60,2
3,18-19,57,55,2
4,19-20,51,51,0
5,20-21,52,52,0
6,21-22,56,56,0
7,22-23,61,59,2
8,23-24,55,55,0
9,24-25,67,61,6



OVERALL RECONCILIATION SUMMARY

player_season_stats
Master records: 386
FBref records: 375
Difference: 11

matches
Master records: 630
FBref records: 617
Difference: 13

MASTER RECONCILIATION AUDIT COMPLETED.
No original master datasets were modified.


In [37]:

# ============================================================
# PHASE 4.5 — RECORD-LEVEL RECONCILIATION
# REAL MADRID | 2015-16 TO 2025-26
# READ-ONLY — NO MASTER DATA MODIFICATION
# ============================================================

import pandas as pd
import re
import unicodedata
from pathlib import Path

BASE = Path("/content/real_madrid_project")

MASTER_DIR = (
    BASE / "Data/Cleaned/"
    "Real_Madrid_Master_Datasets_2026-11-18_UPDATED"
)

AUDIT_DIR = (
    BASE / "Data/Processed/03_fbref_validation"
)

AUDIT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. LOAD THE FOUR REQUIRED DATASETS
# ------------------------------------------------------------

master_players = pd.read_csv(
    MASTER_DIR / "Real_Madrid_Player_Season_Stats.csv"
)

master_matches = pd.read_csv(
    MASTER_DIR / "Real_Madrid_All_Matches.csv"
)

fbref_players = pd.read_csv(
    AUDIT_DIR / "FBref_Player_Season_Stats_Staging.csv"
)

fbref_matches = pd.read_csv(
    AUDIT_DIR / "FBref_Match_Logs_Staging.csv"
)

# ------------------------------------------------------------
# 2. NORMALISATION FUNCTIONS
# ------------------------------------------------------------

def normalize_text(value):

    if pd.isna(value):
        return ""

    value = str(value).strip().casefold()

    value = unicodedata.normalize("NFKD", value)

    value = "".join(
        c for c in value
        if not unicodedata.combining(c)
    )

    value = value.replace("&", " and ")

    value = re.sub(r"[^a-z0-9]+", " ", value)

    return " ".join(value.split())


def normalize_season(value):

    if pd.isna(value):
        return ""

    value = str(value).strip()

    match = re.search(
        r"(\d{2}|\d{4})[-/](\d{2}|\d{4})",
        value
    )

    if match:

        start = match.group(1)[-2:]
        end = match.group(2)[-2:]

        return f"{start}-{end}"

    return value


# ------------------------------------------------------------
# 3. PLAYER RECORD RECONCILIATION
# ------------------------------------------------------------

master_players["season_key"] = (
    master_players["season"].apply(normalize_season)
)

fbref_players["season_key"] = (
    fbref_players["season"].apply(normalize_season)
)

master_players["player_key"] = (
    master_players["player_name"].apply(normalize_text)
)

fbref_players["player_key"] = (
    fbref_players["Player"].apply(normalize_text)
)

# Detect duplicate player-season keys
master_player_duplicates = master_players[
    master_players.duplicated(
        ["season_key", "player_key"],
        keep=False
    )
].copy()

fbref_player_duplicates = fbref_players[
    fbref_players.duplicated(
        ["season_key", "player_key"],
        keep=False
    )
].copy()

# Unique player-season identifiers
mp = master_players[
    ["season_key", "player_key", "player_name"]
].drop_duplicates()

fp = fbref_players[
    ["season_key", "player_key", "Player"]
].drop_duplicates()

player_comparison = mp.merge(
    fp,
    on=["season_key", "player_key"],
    how="outer",
    indicator=True,
    suffixes=("_master", "_fbref")
)

master_only_players = player_comparison[
    player_comparison["_merge"] == "left_only"
].copy()

fbref_only_players = player_comparison[
    player_comparison["_merge"] == "right_only"
].copy()

matched_players = player_comparison[
    player_comparison["_merge"] == "both"
].copy()

# Season-by-season player comparison
player_summary = []

for season in sorted(
    set(mp["season_key"]) | set(fp["season_key"])
):

    m = mp[mp["season_key"] == season]
    f = fp[fp["season_key"] == season]

    matched = player_comparison[
        (player_comparison["season_key"] == season)
        & (player_comparison["_merge"] == "both")
    ]

    m_only = player_comparison[
        (player_comparison["season_key"] == season)
        & (player_comparison["_merge"] == "left_only")
    ]

    f_only = player_comparison[
        (player_comparison["season_key"] == season)
        & (player_comparison["_merge"] == "right_only")
    ]

    player_summary.append({
        "season": season,
        "master_unique_players": len(m),
        "fbref_unique_players": len(f),
        "matched_players": len(matched),
        "master_only": len(m_only),
        "fbref_only": len(f_only)
    })

player_summary_df = pd.DataFrame(player_summary)

print("\n" + "=" * 70)
print("PLAYER RECORD RECONCILIATION")
print("=" * 70)

display(player_summary_df)

print("\nMASTER-ONLY PLAYER RECORDS")
display(master_only_players)

print("\nFBREF-ONLY PLAYER RECORDS")
display(fbref_only_players)

print("\nDUPLICATE MASTER PLAYER KEYS")
display(master_player_duplicates)

print("\nDUPLICATE FBREF PLAYER KEYS")
display(fbref_player_duplicates)

# ------------------------------------------------------------
# 4. MATCH RECORD RECONCILIATION
# ------------------------------------------------------------

def prepare_match_keys(df, source):

    result = df.copy()

    result["season_key"] = (
        result["season"].apply(normalize_season)
    )

    result["opponent_key"] = (
        result["opponent"].apply(normalize_text)
    )

    # Robust date parsing
    try:
        result["date_key"] = pd.to_datetime(
            result["match_date"],
            errors="coerce",
            format="mixed"
        ).dt.strftime("%Y-%m-%d")

    except (TypeError, ValueError):

        result["date_key"] = pd.to_datetime(
            result["match_date"],
            errors="coerce"
        ).dt.strftime("%Y-%m-%d")

    result["source"] = source
    result["_row_id"] = result.index

    result["valid_match_key"] = (
        result["season_key"].ne("")
        & result["opponent_key"].ne("")
        & result["date_key"].notna()
    )

    return result


mm = prepare_match_keys(
    master_matches,
    "MASTER"
)

fm = prepare_match_keys(
    fbref_matches,
    "FBREF"
)

KEYS = [
    "season_key",
    "date_key",
    "opponent_key"
]

master_valid = mm[mm["valid_match_key"]].copy()
fbref_valid = fm[fm["valid_match_key"]].copy()

# Detect duplicate matching keys
master_match_duplicates = master_valid[
    master_valid.duplicated(KEYS, keep=False)
].copy()

fbref_match_duplicates = fbref_valid[
    fbref_valid.duplicated(KEYS, keep=False)
].copy()

# Build unique match keys
master_keys = master_valid[
    KEYS + ["_row_id"]
].drop_duplicates(KEYS)

fbref_keys = fbref_valid[
    KEYS + ["_row_id"]
].drop_duplicates(KEYS)

match_comparison = master_keys.merge(
    fbref_keys,
    on=KEYS,
    how="outer",
    indicator=True,
    suffixes=("_master", "_fbref")
)

# Identify unmatched records
master_only_ids = match_comparison.loc[
    match_comparison["_merge"] == "left_only",
    "_row_id_master"
].dropna().astype(int)

fbref_only_ids = match_comparison.loc[
    match_comparison["_merge"] == "right_only",
    "_row_id_fbref"
].dropna().astype(int)

unmatched_master_matches = mm[
    mm["_row_id"].isin(master_only_ids)
].copy()

unmatched_fbref_matches = fm[
    fm["_row_id"].isin(fbref_only_ids)
].copy()

matched_match_keys = match_comparison[
    match_comparison["_merge"] == "both"
][KEYS]

# Compare competition labels for matched matches
master_comp = master_valid[
    KEYS + ["competition"]
].drop_duplicates(KEYS).rename(
    columns={"competition": "master_competition"}
)

fbref_comp = fbref_valid[
    KEYS + ["competition"]
].drop_duplicates(KEYS).rename(
    columns={"competition": "fbref_competition"}
)

competition_comparison = master_comp.merge(
    fbref_comp,
    on=KEYS,
    how="inner"
)

competition_comparison["competition_match"] = (
    competition_comparison["master_competition"]
    .apply(normalize_text)
    ==
    competition_comparison["fbref_competition"]
    .apply(normalize_text)
)

competition_mismatches = competition_comparison[
    ~competition_comparison["competition_match"]
].copy()

# ------------------------------------------------------------
# 5. SEASON-BY-SEASON MATCH SUMMARY
# ------------------------------------------------------------

match_summary = []

all_seasons = sorted(
    set(master_valid["season_key"])
    | set(fbref_valid["season_key"])
)

for season in all_seasons:

    m = match_comparison[
        match_comparison["season_key"] == season
    ]

    match_summary.append({
        "season": season,
        "master_unique_matches": len(
            master_keys[
                master_keys["season_key"] == season
            ]
        ),
        "fbref_unique_matches": len(
            fbref_keys[
                fbref_keys["season_key"] == season
            ]
        ),
        "matched_matches": (
            m["_merge"] == "both"
        ).sum(),
        "master_only": (
            m["_merge"] == "left_only"
        ).sum(),
        "fbref_only": (
            m["_merge"] == "right_only"
        ).sum()
    })

match_summary_df = pd.DataFrame(match_summary)

print("\n" + "=" * 70)
print("MATCH RECORD RECONCILIATION")
print("=" * 70)

display(match_summary_df)

print("\nMASTER-ONLY MATCHES")
display(
    unmatched_master_matches[
        [
            "season",
            "match_date",
            "competition",
            "opponent",
            "real_madrid_goals_for",
            "real_madrid_goals_against"
        ]
    ]
)

print("\nFBREF-ONLY MATCHES")
display(
    unmatched_fbref_matches[
        [
            "season",
            "match_date",
            "competition",
            "opponent",
            "result"
        ]
    ]
)

print("\nMATCH COMPETITION LABEL DISCREPANCIES")
display(competition_mismatches)

print("\nDUPLICATE MASTER MATCH KEYS")
display(master_match_duplicates[[
    "season", "match_date", "competition", "opponent"
]])

print("\nDUPLICATE FBREF MATCH KEYS")
display(fbref_match_duplicates[[
    "season", "match_date", "competition", "opponent"
]])

# ------------------------------------------------------------
# 6. RECORDS WITH INCOMPLETE MATCH IDENTIFIERS
# ------------------------------------------------------------

invalid_master_matches = mm[
    ~mm["valid_match_key"]
].copy()

invalid_fbref_matches = fm[
    ~fm["valid_match_key"]
].copy()

print("\nMASTER MATCHES WITH INCOMPLETE IDENTIFIERS:",
      len(invalid_master_matches))

print("FBREF MATCHES WITH INCOMPLETE IDENTIFIERS:",
      len(invalid_fbref_matches))

# ------------------------------------------------------------
# 7. SAVE ALL RECONCILIATION REPORTS
# ------------------------------------------------------------

reports = {
    "player_reconciliation_summary.csv": player_summary_df,
    "player_master_only.csv": master_only_players,
    "player_fbref_only.csv": fbref_only_players,
    "player_master_duplicates.csv": master_player_duplicates,
    "player_fbref_duplicates.csv": fbref_player_duplicates,
    "match_reconciliation_summary.csv": match_summary_df,
    "match_master_only.csv": unmatched_master_matches,
    "match_fbref_only.csv": unmatched_fbref_matches,
    "match_competition_mismatches.csv": competition_mismatches,
    "match_master_duplicates.csv": master_match_duplicates,
    "match_fbref_duplicates.csv": fbref_match_duplicates,
    "match_master_invalid_keys.csv": invalid_master_matches,
    "match_fbref_invalid_keys.csv": invalid_fbref_matches
}

for filename, dataframe in reports.items():

    dataframe.to_csv(
        AUDIT_DIR / filename,
        index=False,
        encoding="utf-8-sig"
    )

print("\n" + "=" * 70)
print("PHASE 4.5 RECONCILIATION COMPLETED")
print("=" * 70)

print("Reports saved to:", AUDIT_DIR)
print("Original master datasets remain unchanged.")



PLAYER RECORD RECONCILIATION


,season,master_unique_players,fbref_unique_players,matched_players,master_only,fbref_only
0,15-16,29,29,27,2,2
1,16-17,28,28,26,2,2
2,17-18,32,32,30,2,2
3,18-19,36,36,35,1,1
4,19-20,32,32,30,2,2
5,20-21,37,37,35,2,2
6,21-22,36,36,34,2,2
7,22-23,36,25,22,14,3
8,23-24,38,38,36,2,2
9,24-25,38,38,37,1,1



MASTER-ONLY PLAYER RECORDS


,season_key,player_key,player_name,Player,_merge
18,15-16,luka modric,Luka Modrić,NaN,left_only
23,15-16,mateo kovacic,Mateo Kovačić,NaN,left_only
48,16-17,luka modric,Luka Modrić,NaN,left_only
54,16-17,mateo kovacic,Mateo Kovačić,NaN,left_only
81,17-18,luka modric,Luka Modrić,NaN,left_only
87,17-18,mateo kovacic,Mateo Kovačić,NaN,left_only
120,18-19,luka modric,Luka Modrić,NaN,left_only
152,19-20,luka jovic,Luka Jović,NaN,left_only
154,19-20,luka modric,Luka Modrić,NaN,left_only
184,20-21,luka jovic,Luka Jović,NaN,left_only



FBREF-ONLY PLAYER RECORDS


,season_key,player_key,player_name,Player,_merge
17,15-16,luka modria,NaN,Luka ModriÄ,right_only
22,15-16,mateo kovaa ia,NaN,Mateo KovaÄiÄ,right_only
47,16-17,luka modria,NaN,Luka ModriÄ,right_only
53,16-17,mateo kovaa ia,NaN,Mateo KovaÄiÄ,right_only
80,17-18,luka modria,NaN,Luka ModriÄ,right_only
86,17-18,mateo kovaa ia,NaN,Mateo KovaÄiÄ,right_only
119,18-19,luka modria,NaN,Luka ModriÄ,right_only
151,19-20,luka jovia,NaN,Luka JoviÄ,right_only
153,19-20,luka modria,NaN,Luka ModriÄ,right_only
183,20-21,luka jovia,NaN,Luka JoviÄ,right_only


... (13 outputs left out)

,season,match_date,competition,opponent



DUPLICATE FBREF MATCH KEYS


,season,match_date,competition,opponent



MASTER MATCHES WITH INCOMPLETE IDENTIFIERS: 0
FBREF MATCHES WITH INCOMPLETE IDENTIFIERS: 1

PHASE 4.5 RECONCILIATION COMPLETED
Reports saved to: /content/real_madrid_project/Data/Processed/03_fbref_validation
Original master datasets remain unchanged.


In [38]:

# ============================================================
# PHASE 4.6 — ENCODING CORRECTION & REVISED RECONCILIATION
# READ-ONLY MASTER DATA AUDIT
# ============================================================

import pandas as pd
import re
import unicodedata

# ------------------------------------------------------------
# 1. REPAIR MOJIBAKE / INCORRECT TEXT ENCODING
# ------------------------------------------------------------

def repair_text(value):

    if pd.isna(value):
        return value

    text = str(value)

    for _ in range(2):

        try:
            repaired = text.encode("latin1").decode("utf-8")
        except (UnicodeEncodeError, UnicodeDecodeError):
            break

        if repaired == text:
            break

        text = repaired

    return text


def normalize_text(value):

    if pd.isna(value):
        return ""

    value = repair_text(value)

    value = str(value).strip().casefold()

    value = unicodedata.normalize("NFKD", value)

    value = "".join(
        c for c in value
        if not unicodedata.combining(c)
    )

    value = value.replace("&", " and ")

    value = re.sub(r"[^a-z0-9]+", " ", value)

    return " ".join(value.split())


def normalize_season(value):

    if pd.isna(value):
        return ""

    value = str(value).strip()

    match = re.search(
        r"(\d{2}|\d{4})[-/](\d{2}|\d{4})",
        value
    )

    if match:
        return (
            f"{match.group(1)[-2:]}-"
            f"{match.group(2)[-2:]}"
        )

    return value


# ------------------------------------------------------------
# 2. REPAIR FBREF PLAYER NAMES
# ------------------------------------------------------------

fbref_players["Player_clean"] = (
    fbref_players["Player"].apply(repair_text)
)

master_players["Player_clean"] = (
    master_players["player_name"].apply(repair_text)
)

# Remove aggregate rows, not actual footballers
aggregate_names = {
    "opponent total",
    "squad total"
}

fbref_players_clean = fbref_players[
    ~fbref_players["Player_clean"]
    .apply(normalize_text)
    .isin(aggregate_names)
].copy()

# Generate consistent keys
for df in [master_players, fbref_players_clean]:

    df["season_key"] = (
        df["season"].apply(normalize_season)
    )

    df["player_key"] = (
        df["Player_clean"].apply(normalize_text)
    )

# ------------------------------------------------------------
# 3. RECONCILE PLAYER RECORDS AGAIN
# ------------------------------------------------------------

mp = master_players[
    ["season_key", "player_key", "Player_clean"]
].drop_duplicates()

fp = fbref_players_clean[
    ["season_key", "player_key", "Player_clean"]
].drop_duplicates()

player_comparison_v2 = mp.merge(
    fp,
    on=["season_key", "player_key"],
    how="outer",
    indicator=True,
    suffixes=("_master", "_fbref")
)

master_only_players_v2 = player_comparison_v2[
    player_comparison_v2["_merge"] == "left_only"
].copy()

fbref_only_players_v2 = player_comparison_v2[
    player_comparison_v2["_merge"] == "right_only"
].copy()

player_summary_v2 = []

for season in sorted(
    set(mp["season_key"]) | set(fp["season_key"])
):

    comparison = player_comparison_v2[
        player_comparison_v2["season_key"] == season
    ]

    player_summary_v2.append({
        "season": season,
        "master_players": len(
            mp[mp["season_key"] == season]
        ),
        "fbref_players": len(
            fp[fp["season_key"] == season]
        ),
        "matched": (
            comparison["_merge"] == "both"
        ).sum(),
        "master_only": (
            comparison["_merge"] == "left_only"
        ).sum(),
        "fbref_only": (
            comparison["_merge"] == "right_only"
        ).sum()
    })

player_summary_v2 = pd.DataFrame(player_summary_v2)

print("\n" + "=" * 75)
print("REVISED PLAYER RECONCILIATION")
print("=" * 75)

display(player_summary_v2)

print("\nMASTER-ONLY PLAYERS")
display(master_only_players_v2)

print("\nFBREF-ONLY PLAYERS")
display(fbref_only_players_v2)

# Special investigation: 2022-23
print("\n" + "=" * 75)
print("2022-23 PLAYER COVERAGE INVESTIGATION")
print("=" * 75)

print("\nMASTER 2022-23 PLAYERS")

display(
    mp[mp["season_key"] == "22-23"]
    .sort_values("player_key")
)

print("\nFBREF 2022-23 PLAYERS")

display(
    fp[fp["season_key"] == "22-23"]
    .sort_values("player_key")
)

# ------------------------------------------------------------
# 4. REPAIR FBREF MATCH OPPONENT NAMES
# ------------------------------------------------------------

mm = master_matches.copy()
fm = fbref_matches.copy()

mm["opponent_clean"] = (
    mm["opponent"].apply(repair_text)
)

fm["opponent_clean"] = (
    fm["opponent"].apply(repair_text)
)

for df in [mm, fm]:

    df["season_key"] = (
        df["season"].apply(normalize_season)
    )

    df["opponent_key"] = (
        df["opponent_clean"].apply(normalize_text)
    )

    df["date_key"] = pd.to_datetime(
        df["match_date"],
        errors="coerce",
        format="mixed"
    ).dt.strftime("%Y-%m-%d")

    df["valid_key"] = (
        df["season_key"].ne("")
        & df["opponent_key"].ne("")
        & df["date_key"].notna()
    )

    df["row_id"] = df.index

# ------------------------------------------------------------
# 5. IDENTIFY INVALID FBREF MATCH RECORD
# ------------------------------------------------------------

invalid_fbref_v2 = fm[
    ~fm["valid_key"]
].copy()

print("\nFBREF MATCHES WITH INCOMPLETE IDENTIFIERS")

display(
    invalid_fbref_v2
)

# ------------------------------------------------------------
# 6. REVISED MATCH RECONCILIATION
# ------------------------------------------------------------

KEYS = [
    "season_key",
    "date_key",
    "opponent_key"
]

mm_valid = mm[mm["valid_key"]].copy()
fm_valid = fm[fm["valid_key"]].copy()

master_match_keys_v2 = mm_valid[
    KEYS
].drop_duplicates()

fbref_match_keys_v2 = fm_valid[
    KEYS
].drop_duplicates()

match_comparison_v2 = master_match_keys_v2.merge(
    fbref_match_keys_v2,
    on=KEYS,
    how="outer",
    indicator=True
)

master_only_match_keys_v2 = match_comparison_v2[
    match_comparison_v2["_merge"] == "left_only"
].copy()

fbref_only_match_keys_v2 = match_comparison_v2[
    match_comparison_v2["_merge"] == "right_only"
].copy()

# Retrieve complete original records
master_only_matches_v2 = master_only_match_keys_v2.merge(
    mm_valid,
    on=KEYS,
    how="left"
)

fbref_only_matches_v2 = fbref_only_match_keys_v2.merge(
    fm_valid,
    on=KEYS,
    how="left"
)

match_summary_v2 = []

for season in sorted(
    set(mm_valid["season_key"])
    | set(fm_valid["season_key"])
):

    comparison = match_comparison_v2[
        match_comparison_v2["season_key"] == season
    ]

    match_summary_v2.append({
        "season": season,
        "master_matches": len(
            master_match_keys_v2[
                master_match_keys_v2["season_key"] == season
            ]
        ),
        "fbref_matches": len(
            fbref_match_keys_v2[
                fbref_match_keys_v2["season_key"] == season
            ]
        ),
        "matched": (
            comparison["_merge"] == "both"
        ).sum(),
        "master_only": (
            comparison["_merge"] == "left_only"
        ).sum(),
        "fbref_only": (
            comparison["_merge"] == "right_only"
        ).sum()
    })

match_summary_v2 = pd.DataFrame(match_summary_v2)

print("\n" + "=" * 75)
print("REVISED MATCH RECONCILIATION")
print("=" * 75)

display(match_summary_v2)

print("\nMASTER-ONLY MATCHES")

display(
    master_only_matches_v2[
        [
            "season",
            "match_date",
            "competition",
            "opponent"
        ]
    ]
)

print("\nFBREF-ONLY MATCHES")

display(
    fbref_only_matches_v2[
        [
            "season",
            "match_date",
            "competition",
            "opponent"
        ]
    ]
)

# ------------------------------------------------------------
# 7. COMPETITION LABEL STANDARDISATION
# ------------------------------------------------------------

def normalize_competition(value):

    key = normalize_text(value)

    aliases = {
        "champions lg": "uefa champions league",
        "uefa champions league": "uefa champions league",
        "europa lg": "uefa europa league",
        "uefa europa league": "uefa europa league",
        "fifa club world cup": "fifa club world cup"
    }

    return aliases.get(key, key)


print("\n" + "=" * 75)
print("COMPETITION LABEL AUDIT")
print("=" * 75)

mm_valid["competition_key"] = (
    mm_valid["competition"].apply(normalize_competition)
)

fm_valid["competition_key"] = (
    fm_valid["competition"].apply(normalize_competition)
)

print(
    "Master competition labels:",
    mm_valid["competition_key"].nunique()
)

print(
    "FBref competition labels:",
    fm_valid["competition_key"].nunique()
)

# ------------------------------------------------------------
# 8. SAVE REVISED AUDIT REPORTS
# ------------------------------------------------------------

reports_v2 = {
    "player_reconciliation_summary_v2.csv":
        player_summary_v2,

    "player_master_only_v2.csv":
        master_only_players_v2,

    "player_fbref_only_v2.csv":
        fbref_only_players_v2,

    "match_reconciliation_summary_v2.csv":
        match_summary_v2,

    "match_master_only_v2.csv":
        master_only_matches_v2,

    "match_fbref_only_v2.csv":
        fbref_only_matches_v2,

    "fbref_invalid_match_records_v2.csv":
        invalid_fbref_v2
}

for filename, df in reports_v2.items():

    df.to_csv(
        AUDIT_DIR / filename,
        index=False,
        encoding="utf-8-sig"
    )

print("\n" + "=" * 75)
print("PHASE 4.6 COMPLETED")
print("=" * 75)

print("Reports:", AUDIT_DIR)
print("Master datasets modified: NO")



REVISED PLAYER RECONCILIATION


,season,master_players,fbref_players,matched,master_only,fbref_only
0,15-16,29,29,29,0,0
1,16-17,28,28,28,0,0
2,17-18,32,32,32,0,0
3,18-19,36,36,36,0,0
4,19-20,32,32,32,0,0
5,20-21,37,37,37,0,0
6,21-22,36,36,36,0,0
7,22-23,36,23,23,13,0
8,23-24,38,38,38,0,0
9,24-25,38,38,38,0,0



MASTER-ONLY PLAYERS


,season_key,player_key,Player_clean_master,Player_clean_fbref,_merge
231,22-23,alvaro rodriguez,Álvaro Rodríguez,NaN,left_only
234,22-23,aurelien tchouameni,Aurélien Tchouaméni,NaN,left_only
235,22-23,carlos dotor,Carlos Dotor,NaN,left_only
236,22-23,casemiro,Casemiro,NaN,left_only
239,22-23,david alaba,David Alaba,NaN,left_only
240,22-23,diego pineiro,Diego Piñeiro,NaN,left_only
246,22-23,fran gonzalez,Fran González,NaN,left_only
249,22-23,lucas canizares,Lucas Cañizares,NaN,left_only
255,22-23,mario de luis,Mario de Luis,NaN,left_only
257,22-23,marvel,Marvel,NaN,left_only



FBREF-ONLY PLAYERS


,season_key,player_key,Player_clean_master,Player_clean_fbref,_merge


... (9 outputs left out)

,season,match_date,competition,opponent
0,2016-17,2016-12-15,FIFA Club World Cup,CF América
1,2016-17,2016-12-18,FIFA Club World Cup,Kashima Antlers
2,2017-18,2017-12-13,FIFA Club World Cup,Al Jazira Club
3,2017-18,2017-12-16,FIFA Club World Cup,Grêmio Porto Alegre
4,2018-19,2018-12-19,FIFA Club World Cup,Kashima Antlers
5,2018-19,2018-12-22,FIFA Club World Cup,Al Ain FC
6,2022-23,2023-02-08,FIFA Club World Cup,Al Ahly SC
7,2022-23,2023-02-11,FIFA Club World Cup,Al Hilal
8,2024-25,2025-06-18,FIFA Club World Cup,Al Hilal
9,2024-25,2025-06-22,FIFA Club World Cup,CF Pachuca



FBREF-ONLY MATCHES


,season,match_date,competition,opponent



COMPETITION LABEL AUDIT
Master competition labels: 6
FBref competition labels: 5

PHASE 4.6 COMPLETED
Reports: /content/real_madrid_project/Data/Processed/03_fbref_validation
Master datasets modified: NO


In [39]:

# ============================================================
# PHASE 4.7 — 2022-23 PLAYER & COMPETITION COVERAGE
# DIAGNOSTIC ONLY — NO MASTER DATA MODIFICATIONS
# ============================================================

# 1. Inspect original FBref player records for 2022-23

fbref_2223_raw = fbref_players[
    fbref_players["season"].apply(normalize_season) == "22-23"
].copy()

print("=" * 75)
print("FBREF 2022-23 PLAYER SOURCE")
print("=" * 75)

print("Total records:", len(fbref_2223_raw))
print("Unique players:", fbref_2223_raw["Player"].nunique())

print("\nAvailable columns:")
print(fbref_2223_raw.columns.tolist())

# Show source distribution if available
if "source_file" in fbref_2223_raw.columns:

    print("\nSource file distribution:")

    display(
        fbref_2223_raw.groupby(
            "source_file"
        ).agg(
            records=("Player", "size"),
            players=("Player", "nunique")
        ).reset_index()
    )

# Show all original FBref records
print("\nAll 2022-23 FBref player records:")

display(
    fbref_2223_raw.sort_values("Player")
)

# ------------------------------------------------------------
# 2. Investigate all 13 unmatched players
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("13 MASTER-ONLY PLAYERS — SOURCE INVESTIGATION")
print("=" * 75)

missing_keys = master_only_players_v2[
    "player_key"
].tolist()

missing_in_source = []

for key in missing_keys:

    records = fbref_players[
        fbref_players["Player"].apply(normalize_text) == key
    ].copy()

    for _, row in records.iterrows():

        missing_in_source.append({
            "player_key": key,
            "fbref_name": row["Player"],
            "season": row["season"],
            "source_file": row.get("source_file", None)
        })

if missing_in_source:

    display(pd.DataFrame(missing_in_source))

else:

    print("No corresponding FBref records found in other seasons.")

# ------------------------------------------------------------
# 3. Inspect exact competition labels
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("MASTER COMPETITION LABELS")
print("=" * 75)

display(
    mm["competition"]
    .value_counts(dropna=False)
    .rename_axis("competition")
    .reset_index(name="matches")
)

print("\n" + "=" * 75)
print("FBREF COMPETITION LABELS")
print("=" * 75)

display(
    fm["competition"]
    .value_counts(dropna=False)
    .rename_axis("competition")
    .reset_index(name="matches")
)

# ------------------------------------------------------------
# 4. Compare 2022-23 match coverage by competition
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("2022-23 MASTER MATCHES BY COMPETITION")
print("=" * 75)

display(
    mm[mm["season_key"] == "22-23"]
    ["competition"]
    .value_counts()
    .rename_axis("competition")
    .reset_index(name="matches")
)

print("\n2022-23 FBREF MATCHES BY COMPETITION")

display(
    fm[fm["season_key"] == "22-23"]
    ["competition"]
    .value_counts()
    .rename_axis("competition")
    .reset_index(name="matches")
)

# ------------------------------------------------------------
# 5. Final diagnostic status
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("PHASE 4.7 DIAGNOSTIC COMPLETED")
print("=" * 75)

print("Player discrepancy:", len(missing_keys))
print("Invalid FBref match rows:", len(invalid_fbref_v2))
print("Master-only match records:", len(master_only_matches_v2))

print("Master datasets modified: NO")


FBREF 2022-23 PLAYER SOURCE
Total records: 25
Unique players: 25

Available columns:
['Player', 'Nation', 'Pos', 'Age', 'MP', 'Starts', 'Minutes', 'Nineties', 'Goals', 'Assists', 'Goals_Assists', 'Non_Penalty_Goals', 'Penalty_Goals', 'Penalty_Attempts', 'Yellow_Cards', 'Red_Cards', 'Goals_Per_90', 'Assists_Per_90', 'Goals_Assists_Per_90', 'Non_Penalty_Goals_Per_90', 'Non_Penalty_Goals_Assists_Per_90', 'Matches', 'season', 'source_file', 'season_key', 'player_key', 'Player_clean']

Source file distribution:


,source_file,records,players
0,sportsref_download stats 22-23.xls,25,25



All 2022-23 FBref player records:


,Player,Nation,Pos,Age,MP,Starts,Minutes,Nineties,Goals,Assists,...,Assists_Per_90,Goals_Assists_Per_90,Non_Penalty_Goals_Per_90,Non_Penalty_Goals_Assists_Per_90,Matches,season,source_file,season_key,player_key,Player_clean
18,Andriy Lunin,ua UKR,GK,23.0,0,0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Matches,22-23,sportsref_download stats 22-23.xls,22-23,andriy lunin,Andriy Lunin
3,Antonio Rüdiger,de GER,DF,29.0,2,2,210.0,2.3,0.0,0.0,...,0.00,0.00,0.00,0.00,Matches,22-23,sportsref_download stats 22-23.xls,22-23,antonio rudiger,Antonio Rüdiger
12,Dani Carvajal,es ESP,DF,30.0,2,1,124.0,1.4,0.0,0.0,...,0.00,0.00,0.00,0.00,Matches,22-23,sportsref_download stats 22-23.xls,22-23,dani carvajal,Dani Carvajal
15,Dani Ceballos,es ESP,MF,25.0,2,0,42.0,0.5,0.0,0.0,...,0.00,0.00,0.00,0.00,Matches,22-23,sportsref_download stats 22-23.xls,22-23,dani ceballos,Dani Ceballos
16,Eden Hazard,be BEL,"FW,MF",31.0,0,0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Matches,22-23,sportsref_download stats 22-23.xls,22-23,eden hazard,Eden Hazard
7,Eduardo Camavinga,fr FRA,MF,19.0,2,2,90.0,1.0,0.0,0.0,...,0.00,0.00,0.00,0.00,Matches,22-23,sportsref_download stats 22-23.xls,22-23,eduardo camavinga,Eduardo Camavinga
4,Federico Valverde,uy URU,"MF,FW",24.0,2,2,194.0,2.2,0.0,0.0,...,0.00,0.00,0.00,0.00,Matches,22-23,sportsref_download stats 22-23.xls,22-23,federico valverde,Federico Valverde
10,Ferland Mendy,fr FRA,DF,27.0,2,1,137.0,1.5,0.0,0.0,...,0.00,0.00,0.00,0.00,Matches,22-23,sportsref_download stats 22-23.xls,22-23,ferland mendy,Ferland Mendy
22,Jesús Vallejo,es ESP,DF,25.0,0,0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Matches,22-23,sportsref_download stats 22-23.xls,22-23,jesus vallejo,Jesús Vallejo
0,Karim Benzema,fr FRA,FW,34.0,2,2,210.0,2.3,2.0,0.0,...,0.00,0.86,0.43,0.43,Matches,22-23,sportsref_download stats 22-23.xls,22-23,karim benzema,Karim Benzema



13 MASTER-ONLY PLAYERS — SOURCE INVESTIGATION


,player_key,fbref_name,season,source_file
0,alvaro rodriguez,Álvaro Rodríguez,23-24,sportsref_download stats 23-24.xls
1,aurelien tchouameni,Aurélien Tchouaméni,23-24,sportsref_download stats 23-24.xls
2,aurelien tchouameni,Aurélien Tchouaméni,24-25,sportsref_download stats 24-25.xls
3,aurelien tchouameni,Aurélien Tchouaméni,25-26,sportsref_download stats 25-26.xls
4,casemiro,Casemiro,15-16,sportsref_download stats 15-16.xls
5,casemiro,Casemiro,16-17,sportsref_download stats 16-17.xls
6,casemiro,Casemiro,17-18,sportsref_download stats 17-18.xls
7,casemiro,Casemiro,18-19,sportsref_download stats 18-19.xls
8,casemiro,Casemiro,19-20,sportsref_download stats 19-20.xls
9,casemiro,Casemiro,20-21,sportsref_download stats 20-21.xls


... (5 outputs left out)

,competition,matches
0,La Liga,38
1,UEFA Champions League,12
2,Copa del Rey,6
3,Spanish Super Cup,2
4,FIFA Club World Cup,2
5,UEFA Super Cup,1



2022-23 FBREF MATCHES BY COMPETITION


,competition,matches
0,La Liga,38
1,Champions Lg,12
2,Copa del Rey,6
3,Supercopa de España,2
4,Super Cup,1



PHASE 4.7 DIAGNOSTIC COMPLETED
Player discrepancy: 13
Invalid FBref match rows: 1
Master-only match records: 14
Master datasets modified: NO


In [40]:

import pandas as pd
import unicodedata
import re

from google.colab import files

uploaded = files.upload()

fbref_2223 = pd.read_csv(
    "Real_Madrid_FBref_2022-23_Standard_Stats_Staging.csv"
)

def clean_name(x):
    x = str(x).casefold().strip()
    x = unicodedata.normalize("NFKD", x)
    x = "".join(
        c for c in x
        if not unicodedata.combining(c)
    )
    x = re.sub(r"[^a-z0-9]+", " ", x)
    return " ".join(x.split())

master_2223 = master_players[
    master_players["season"].apply(normalize_season) == "22-23"
].copy()

master_2223["player_key"] = (
    master_2223["player_name"].apply(clean_name)
)

fbref_2223["player_key"] = (
    fbref_2223["Player"].apply(clean_name)
)

master_keys = master_2223[
    ["player_key"]
].drop_duplicates()

fbref_keys = fbref_2223[
    ["player_key"]
].drop_duplicates()

comparison = master_keys.merge(
    fbref_keys,
    on="player_key",
    how="outer",
    indicator=True
)

print("PHASE 4.8 — FINAL 2022-23 PLAYER AUDIT")

print("\nReconciliation:")
display(comparison["_merge"].value_counts())

print("\nRemaining master-only players:")
display(
    comparison[
        comparison["_merge"] == "left_only"
    ]
)

print("\nRemaining FBref-only players:")
display(
    comparison[
        comparison["_merge"] == "right_only"
    ]
)

print("\nMaster records:", len(master_keys))
print("FBref records:", len(fbref_keys))
print("Master datasets modified: NO")


Saving Real_Madrid_FBref_2022-23_Standard_Stats_Staging.csv to Real_Madrid_FBref_2022-23_Standard_Stats_Staging.csv
PHASE 4.8 — FINAL 2022-23 PLAYER AUDIT

Reconciliation:


,count
_merge,
both,36
left_only,0
right_only,0



Remaining master-only players:


,player_key,_merge



Remaining FBref-only players:


,player_key,_merge



Master records: 36
FBref records: 36
Master datasets modified: NO


In [41]:

# ============================================================
# PHASE 4.9 — 2022-23 NUMERICAL STATISTICS AUDIT
# ============================================================

print("MASTER DATASET COLUMNS")
print(master_2223.columns.tolist())

print("\nFBREF DATASET COLUMNS")
print(fbref_2223.columns.tolist())

print("\nMASTER SAMPLE")
display(master_2223.head(5))

print("\nFBREF SAMPLE")
display(fbref_2223.head(5))

# Check duplicate player identifiers
print("\nMASTER DUPLICATE PLAYER KEYS:",
      master_2223["player_key"].duplicated().sum())

print("FBREF DUPLICATE PLAYER KEYS:",
      fbref_2223["player_key"].duplicated().sum())

# Numeric field availability
print("\nFBREF NUMERIC FIELDS")

display(
    fbref_2223.select_dtypes(
        include="number"
    ).describe().T
)

print("\nPHASE 4.9 SCHEMA INSPECTION COMPLETE")


MASTER DATASET COLUMNS
['player_id', 'player_key_candidate', 'player_name', 'nation', 'position', 'age', 'season', 'season_role', 'competition_scope', 'appearances', 'starts', 'minutes', 'nineties', 'goals', 'assists', 'goal_contributions', 'non_penalty_goals', 'penalties_scored', 'penalties_attempted', 'yellow_cards', 'red_cards', 'goals_per90', 'assists_per90', 'goal_contributions_per90', 'non_penalty_goals_per90', 'goal_contributions_minus_penalties_per90', 'source_name', 'source_file', 'source_row', 'season_key', 'player_key', 'Player_clean']

FBREF DATASET COLUMNS
['Player', 'Nation', 'Pos', 'Age', 'MP', 'Starts', 'Min', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'CrdY', 'CrdR', 'Gls_per90', 'Ast_per90', 'G+A_per90', 'G-PK_per90', 'G+A-PK_per90', 'Matches', 'season', 'source_file', 'source_table', 'player_key']

MASTER SAMPLE


,player_id,player_key_candidate,player_name,nation,position,age,season,season_role,competition_scope,appearances,...,assists_per90,goal_contributions_per90,non_penalty_goals_per90,goal_contributions_minus_penalties_per90,source_name,source_file,source_row,season_key,player_key,Player_clean
0,RMPLR-vinicius-junior,vinicius-junior,Vinicius Júnior,br BRA,FW,22,2022-23,Main,All competitions,53,...,0.37,0.77,0.39,0.77,FBref / Sports Reference,sportsref_download.xls (corrected 2022-23),1,22-23,vinicius junior,Vinicius Júnior
1,RMPLR-thibaut-courtois,thibaut-courtois,Thibaut Courtois,be BEL,GK,30,2022-23,Main,All competitions,49,...,0.00,0.00,0.00,0.00,FBref / Sports Reference,sportsref_download.xls (corrected 2022-23),2,22-23,thibaut courtois,Thibaut Courtois
2,RMPLR-eder-militao,eder-militao,Éder Militão,br BRA,DF,24,2022-23,Main,All competitions,51,...,0.00,0.15,0.15,0.15,FBref / Sports Reference,sportsref_download.xls (corrected 2022-23),3,22-23,eder militao,Éder Militão
3,RMPLR-federico-valverde,federico-valverde,Federico Valverde,uy URU,"MF,FW",24,2022-23,Main,All competitions,54,...,0.15,0.34,0.19,0.34,FBref / Sports Reference,sportsref_download.xls (corrected 2022-23),4,22-23,federico valverde,Federico Valverde
4,RMPLR-rodrygo,rodrygo,Rodrygo,br BRA,FW,21,2022-23,Main,All competitions,55,...,0.24,0.68,0.36,0.60,FBref / Sports Reference,sportsref_download.xls (corrected 2022-23),5,22-23,rodrygo,Rodrygo



FBREF SAMPLE


,Player,Nation,Pos,Age,MP,Starts,Min,90s,Gls,Ast,...,Gls_per90,Ast_per90,G+A_per90,G-PK_per90,G+A-PK_per90,Matches,season,source_file,source_table,player_key
0,Vinicius Júnior,br BRA,FW,22,53,51,4564.0,50.7,20.0,19.0,...,0.39,0.37,0.77,0.39,0.77,Matches,2022-23,22-23.xls,FBref Standard Stats,vinicius junior
1,Thibaut Courtois,be BEL,GK,30,49,49,4470.0,49.7,0.0,0.0,...,0.00,0.00,0.00,0.00,0.00,Matches,2022-23,22-23.xls,FBref Standard Stats,thibaut courtois
2,Éder Militão,br BRA,DF,24,51,48,4228.0,47.0,7.0,0.0,...,0.15,0.00,0.15,0.15,0.15,Matches,2022-23,22-23.xls,FBref Standard Stats,eder militao
3,Federico Valverde,uy URU,"MF,FW",24,54,48,4211.0,46.8,9.0,7.0,...,0.19,0.15,0.34,0.19,0.34,Matches,2022-23,22-23.xls,FBref Standard Stats,federico valverde
4,Rodrygo,br BRA,FW,21,55,40,3725.0,41.4,18.0,10.0,...,0.43,0.24,0.68,0.36,0.60,Matches,2022-23,22-23.xls,FBref Standard Stats,rodrygo



MASTER DUPLICATE PLAYER KEYS: 0
FBREF DUPLICATE PLAYER KEYS: 0

FBREF NUMERIC FIELDS


,count,mean,std,min,25%,50%,75%,max
Age,36.0,24.722222,5.300195,17.0,20.00,24.00,30.00,36.00
MP,36.0,24.638889,22.938617,0.0,0.75,19.00,49.25,57.00
Starts,36.0,18.027778,18.969880,0.0,0.00,13.00,35.50,51.00
Min,27.0,2186.592593,1617.412493,6.0,279.00,2578.00,3602.00,4564.00
90s,27.0,24.300000,17.968755,0.1,3.10,28.60,40.05,50.70
Gls,27.0,4.296296,7.513141,0.0,0.00,1.00,5.00,30.00
Ast,27.0,3.296296,4.312904,0.0,0.00,2.00,5.50,19.00
G+A,27.0,7.592593,10.867238,0.0,0.50,3.00,8.00,39.00
G-PK,27.0,3.777778,6.210000,0.0,0.00,1.00,4.00,21.00
PK,27.0,0.518519,1.826522,0.0,0.00,0.00,0.00,9.00



PHASE 4.9 SCHEMA INSPECTION COMPLETE


In [42]:

# ============================================================
# PHASE 4.9B — MASTER VS FBREF NUMERICAL RECONCILIATION
# READ-ONLY AUDIT
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# Master-to-FBref column mapping
column_map = {
    "appearances": "MP",
    "starts": "Starts",
    "minutes": "Min",
    "nineties": "90s",
    "goals": "Gls",
    "assists": "Ast",
    "goal_contributions": "G+A",
    "non_penalty_goals": "G-PK",
    "penalties_scored": "PK",
    "penalties_attempted": "PKatt",
    "yellow_cards": "CrdY",
    "red_cards": "CrdR",
    "goals_per90": "Gls_per90",
    "assists_per90": "Ast_per90",
    "goal_contributions_per90": "G+A_per90",
    "non_penalty_goals_per90": "G-PK_per90",
    "goal_contributions_minus_penalties_per90": "G+A-PK_per90"
}

# Ensure unique player keys
assert not master_2223["player_key"].duplicated().any()
assert not fbref_2223["player_key"].duplicated().any()

# Merge
audit = master_2223.merge(
    fbref_2223,
    on="player_key",
    how="outer",
    indicator=True,
    suffixes=("_master", "_fbref"),
    validate="one_to_one"
)

assert (audit["_merge"] == "both").all(), \
    "Player identities do not fully match."

results = []

# Compare every metric
for master_col, fbref_col in column_map.items():

    master_values = pd.to_numeric(
        audit[master_col], errors="coerce"
    )

    fbref_values = pd.to_numeric(
        audit[fbref_col], errors="coerce"
    )

    # Allow for source rounding
    tolerance = (
        0.051 if master_col == "nineties"
        else 0.011 if "per90" in master_col
        else 0.5
    )

    for i in range(len(audit)):

        m = master_values.iloc[i]
        f = fbref_values.iloc[i]

        if pd.isna(m) and pd.isna(f):
            status = "BOTH_MISSING"
            difference = np.nan

        elif pd.isna(m):
            status = "MASTER_MISSING"
            difference = np.nan

        elif pd.isna(f):
            status = "FBREF_MISSING"
            difference = np.nan

        else:
            difference = m - f

            status = (
                "MATCH"
                if abs(difference) <= tolerance
                else "MISMATCH"
            )

        results.append({
            "player_name": audit.iloc[i]["player_name"],
            "player_key": audit.iloc[i]["player_key"],
            "metric": master_col,
            "master_value": m,
            "fbref_value": f,
            "difference": difference,
            "status": status
        })

comparison = pd.DataFrame(results)

# ------------------------------------------------------------
# SUMMARY BY METRIC
# ------------------------------------------------------------

summary = (
    comparison.groupby(["metric", "status"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

print("=" * 75)
print("PHASE 4.9B — STATISTICAL ACCURACY RESULTS")
print("=" * 75)

display(summary)

# ------------------------------------------------------------
# ACTUAL DISCREPANCIES
# ------------------------------------------------------------

mismatches = comparison[
    comparison["status"] == "MISMATCH"
].copy()

print("\nNUMERICAL MISMATCHES:", len(mismatches))

display(mismatches)

# ------------------------------------------------------------
# MISSING VALUE INVESTIGATION
# ------------------------------------------------------------

missing_stats = comparison[
    comparison["status"].isin([
        "MASTER_MISSING",
        "FBREF_MISSING",
        "BOTH_MISSING"
    ])
].copy()

print("\nMISSING STATISTICAL VALUES")

display(missing_stats)

# ------------------------------------------------------------
# ZERO APPEARANCE INVESTIGATION
# ------------------------------------------------------------

zero_appearance = audit[
    pd.to_numeric(audit["MP"], errors="coerce") == 0
].copy()

print("\nZERO-APPEARANCE PLAYERS:", len(zero_appearance))

display(
    zero_appearance[
        [
            "Player",
            "MP",
            "Starts",
            "Min",
            "Gls",
            "Ast"
        ]
    ]
)

# ------------------------------------------------------------
# SAVE REPORTS
# ------------------------------------------------------------

output_dir = Path(
    "/content/real_madrid_project/Data/Processed/"
    "03_fbref_validation"
)

output_dir.mkdir(parents=True, exist_ok=True)

comparison.to_csv(
    output_dir / "2022_23_statistical_comparison.csv",
    index=False
)

summary.to_csv(
    output_dir / "2022_23_statistical_audit_summary.csv",
    index=False
)

mismatches.to_csv(
    output_dir / "2022_23_statistical_mismatches.csv",
    index=False
)

missing_stats.to_csv(
    output_dir / "2022_23_missing_statistics.csv",
    index=False
)

print("\nPHASE 4.9B COMPLETED")
print("Reports saved:", output_dir)
print("MASTER DATASETS MODIFIED: NO")


PHASE 4.9B — STATISTICAL ACCURACY RESULTS


status,metric,BOTH_MISSING,MATCH
0,appearances,0,36
1,assists,9,27
2,assists_per90,9,27
3,goal_contributions,9,27
4,goal_contributions_minus_penalties_per90,9,27
5,goal_contributions_per90,9,27
6,goals,9,27
7,goals_per90,9,27
8,minutes,9,27
9,nineties,9,27



NUMERICAL MISMATCHES: 0


,player_name,player_key,metric,master_value,fbref_value,difference,status



MISSING STATISTICAL VALUES


,player_name,player_key,metric,master_value,fbref_value,difference,status
77,Carlos Dotor,carlos dotor,minutes,NaN,NaN,NaN,BOTH_MISSING
82,Diego Piñeiro,diego pineiro,minutes,NaN,NaN,NaN,BOTH_MISSING
88,Fran González,fran gonzalez,minutes,NaN,NaN,NaN,BOTH_MISSING
91,Lucas Cañizares,lucas canizares,minutes,NaN,NaN,NaN,BOTH_MISSING
93,Luis López,luis lopez,minutes,NaN,NaN,NaN,BOTH_MISSING
...,...,...,...,...,...,...,...
597,Luis López,luis lopez,goal_contributions_minus_penalties_per90,NaN,NaN,NaN,BOTH_MISSING
601,Mario de Luis,mario de luis,goal_contributions_minus_penalties_per90,NaN,NaN,NaN,BOTH_MISSING
603,Marvel,marvel,goal_contributions_minus_penalties_per90,NaN,NaN,NaN,BOTH_MISSING
605,Nico Paz,nico paz,goal_contributions_minus_penalties_per90,NaN,NaN,NaN,BOTH_MISSING



ZERO-APPEARANCE PLAYERS: 9


,Player,MP,Starts,Min,Gls,Ast
5,Carlos Dotor,0,0,NaN,NaN,NaN
10,Diego Piñeiro,0,0,NaN,NaN,NaN
16,Fran González,0,0,NaN,NaN,NaN
19,Lucas Cañizares,0,0,NaN,NaN,NaN
21,Luis López,0,0,NaN,NaN,NaN
25,Mario de Luis,0,0,NaN,NaN,NaN
27,Marvel,0,0,NaN,NaN,NaN
29,Nico Paz,0,0,NaN,NaN,NaN
35,Vinicius Tobias,0,0,NaN,NaN,NaN



PHASE 4.9B COMPLETED
Reports saved: /content/real_madrid_project/Data/Processed/03_fbref_validation
MASTER DATASETS MODIFIED: NO


In [43]:

# ============================================================
# PHASE 4.10 — ZERO APPEARANCE CLASSIFICATION
# READ-ONLY MASTER DATASET
# ============================================================

import pandas as pd
from pathlib import Path

# Identify zero-appearance players
zero_players = audit[
    pd.to_numeric(audit["MP"], errors="coerce") == 0
].copy()

# Define statistical classifications
count_metrics = [
    "MP", "Starts", "Min", "90s",
    "Gls", "Ast", "G+A", "G-PK",
    "PK", "PKatt", "CrdY", "CrdR"
]

rate_metrics = [
    "Gls_per90",
    "Ast_per90",
    "G+A_per90",
    "G-PK_per90",
    "G+A-PK_per90"
]

# Create a separate report
classification = zero_players[
    ["Player", "Nation", "Pos", "Age", "MP", "Starts"]
].copy()

classification["classification"] = (
    "Zero first-team appearances"
)

classification["count_metric_treatment"] = (
    "Zero in derived analytical dataset; "
    "preserve source nulls"
)

classification["per90_treatment"] = (
    "Keep NULL — undefined without minutes"
)

classification["master_source_preserved"] = "YES"

print("=" * 70)
print("ZERO-APPEARANCE PLAYER AUDIT")
print("=" * 70)

display(classification)

# Validate all nine records
assert len(classification) == 9

assert (
    pd.to_numeric(classification["MP"]) == 0
).all()

assert (
    pd.to_numeric(classification["Starts"]) == 0
).all()

# Save report separately
output_dir = Path(
    "/content/real_madrid_project/Data/Processed/"
    "03_fbref_validation"
)

classification.to_csv(
    output_dir / "2022_23_zero_appearance_classification.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nZERO-APPEARANCE CLASSIFICATION COMPLETED")
print("Players classified:", len(classification))
print("Master dataset modified: NO")


ZERO-APPEARANCE PLAYER AUDIT


,Player,Nation,Pos,Age,MP,Starts,classification,count_metric_treatment,per90_treatment,master_source_preserved
5,Carlos Dotor,es ESP,MF,21,0,0,Zero first-team appearances,Zero in derived analytical dataset; preserve s...,Keep NULL — undefined without minutes,YES
10,Diego Piñeiro,es ESP,GK,18,0,0,Zero first-team appearances,Zero in derived analytical dataset; preserve s...,Keep NULL — undefined without minutes,YES
16,Fran González,es ESP,DF,23,0,0,Zero first-team appearances,Zero in derived analytical dataset; preserve s...,Keep NULL — undefined without minutes,YES
19,Lucas Cañizares,es ESP,GK,20,0,0,Zero first-team appearances,Zero in derived analytical dataset; preserve s...,Keep NULL — undefined without minutes,YES
21,Luis López,es ESP,GK,21,0,0,Zero first-team appearances,Zero in derived analytical dataset; preserve s...,Keep NULL — undefined without minutes,YES
25,Mario de Luis,es ESP,GK,20,0,0,Zero first-team appearances,Zero in derived analytical dataset; preserve s...,Keep NULL — undefined without minutes,YES
27,Marvel,es ESP,DF,19,0,0,Zero first-team appearances,Zero in derived analytical dataset; preserve s...,Keep NULL — undefined without minutes,YES
29,Nico Paz,ar ARG,MF,17,0,0,Zero first-team appearances,Zero in derived analytical dataset; preserve s...,Keep NULL — undefined without minutes,YES
35,Vinicius Tobias,br BRA,DF,18,0,0,Zero first-team appearances,Zero in derived analytical dataset; preserve s...,Keep NULL — undefined without minutes,YES



ZERO-APPEARANCE CLASSIFICATION COMPLETED
Players classified: 9
Master dataset modified: NO
